## Google Mounting and Libraries download

Connect to Google Drive, allowing files stored in Drive to be assessed, loaded and saved durimg the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Checking for key variables needed for the forecasting pipeline that has been created in the notebnook. check if variable exists before continuing with modelling or evaluation.

In [ ]:
needed_vars = [
    "data",
    "featured_data",
    "splits",
    "feature_cols",
    "target_col",
    "tickers",
    "gdelt_sentiment",
    "stock_features_all",
    "stock_features_sentiment_period",
    "sentiment_data"
]

for var in needed_vars:
    print(var, "exists:", var in globals())

Installing the yfinance library, which is used to collect historical stock market data directly into the Google Colab notebook

In [ ]:
!pip install yfinance --quiet

Importing the main Python libraries used for data handling, numerical calculations, visualisation, date handling and downloading historical stock data from Yahoo Finance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import yfinance as yf
import plotly.io as pio
import yfinance as yf

Checking whether the main datasets, feature variables and GBM simulation function already exist in the notebook before running later modelling or evaluation cells.

In [ ]:
print("data exists:", "data" in globals())
print("featured_data exists:", "featured_data" in globals())
print("splits exists:", "splits" in globals())
print("feature_cols exists:", "feature_cols" in globals())
print("target_col exists:", "target_col" in globals())
print("simulate_gbm exists:", "simulate_gbm" in globals())

##Financial Dataset extraction and  Download

Downloading daily historical stock price data for the selected nine tickers from Yahoo Finance. The cell collects Open, High, Low, Close and Volume data, removes missing values, stores each ticker’s data, and saves the raw stock datasets as CSV files.

In [ ]:
tickers = ["WMT", "NVDA", "AMD", "NFLX", "JNJ", "KO", "MSFT", "AAPL", "AMZN"]  # 3 volatile, 3 stable, 3 medium
start_date = "2005-01-01"
end_date = "2025-12-04"

print("Downloading 20 years of daily OHLCV data...\n")
raw_data = {}

for ticker in tickers:
    print(f"   → Downloading {ticker}...")
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    df.dropna(inplace=True)
    raw_data[ticker] = df
    df.to_csv(f"{ticker}_raw_2005_2025.csv", index_label='Date')
    print(f"      Saved {ticker}_raw_2005_2025.csv → {len(df)} rows")

print("Raw price data collection COMPLETE")

## News Sentiment Collection

Installing the main libraries needed for collecting and processing GDELT/news sentiment data, including BigQuery access, progress tracking, VADER sentiment analysis and Hugging Face model support.

In [ ]:
!pip install google-cloud-bigquery pandas tqdm vaderSentiment huggingface_hub

Importing the libraries needed for GDELT sentiment collection and processing, including file handling, data manipulation, progress tracking, Google Colab authentication, BigQuery access and VADER sentiment analysis.

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from google.colab import auth, files
from google.cloud import bigquery
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

Authenticating the Google Colab session and connecting it to the Google BigQuery project. This allows the notebook to query GDELT news data through BigQuery.

In [ ]:
auth.authenticate_user()

PROJECT_ID = "fyp-smp-project"
client = bigquery.Client(project=PROJECT_ID)

Creating a dictionary of selected stock tickers and their related company search terms. This helps match each stock to relevant GDELT news articles using company names or common keyword variations.

In [ ]:
TICKERS = {
    "WMT": ["walmart"],
    "NVDA": ["nvidia"],
    "AMD": ["amd", "advanced-micro-devices", "advanced micro devices"],
    "JNJ": ["jnj", "johnson-johnson", "johnson and johnson"],
    "KO": ["coca-cola", "coca cola"],
    "MSFT": ["microsoft"],
    "AAPL": ["apple"],
    "AMZN": ["amazon"],
    "NFLX": ["netflix"]
}

TICKER_LIST = list(TICKERS.keys())
TICKER_LIST

Builds a BigQuery query for one ticker and one year, filters GDELT Events by company names, aggregates daily sentiment statistics, and returns the results as a DataFrame

In [ ]:
def get_gdelt_events_sentiment_by_year(ticker, company_names, year):
    conditions = []

    for name in company_names:
        safe_name = name.lower().replace("'", "\\'")
        conditions.append(f"LOWER(SOURCEURL) LIKE '%{safe_name}%'")

    where_clause = " OR ".join(conditions)

    query = f"""
    SELECT
        '{ticker}' AS ticker,
        PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)) AS date,
        COUNT(*) AS news_count,
        AVG(AvgTone) AS sent_mean,
        STDDEV(AvgTone) AS sent_std,
        MIN(AvgTone) AS sent_min,
        MAX(AvgTone) AS sent_max,
        'GDELT_EVENTS' AS source
    FROM `gdelt-bq.gdeltv2.events_partitioned`
    WHERE
        _PARTITIONTIME BETWEEN TIMESTAMP('{year}-01-01') AND TIMESTAMP('{year}-12-31')
        AND SQLDATE BETWEEN {year}0101 AND {year}1231
        AND ({where_clause})
        AND AvgTone IS NOT NULL
    GROUP BY date
    ORDER BY date
    """


    job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=15 * 10**9
    )
    return client.query(query, job_config=job_config).to_dataframe()

Loops through each ticker and each year (2015–2025), runs the GDELT‑query function, and collects all non‑empty yearly results into a single list. It also prints progress and reports any failed queries.

In [ ]:
gdelt_parts = []

for ticker, names in TICKERS.items():
    print(f"\nTicker: {ticker}")

    for year in range(2015, 2026):
        print(f"  Year: {year}")

        try:
            df_year = get_gdelt_events_sentiment_by_year(ticker, names, year)

            if not df_year.empty:
                gdelt_parts.append(df_year)

        except Exception as e:
            print(f"Failed {ticker} {year}: {e}")

Combines all yearly GDELT results into one DataFrame, cleans the date and missing values, removes duplicates, and sorts the sentiment records by ticker and date.

In [ ]:
gdelt_sentiment = pd.concat(gdelt_parts, ignore_index=True)

gdelt_sentiment["date"] = pd.to_datetime(gdelt_sentiment["date"])
gdelt_sentiment["sent_std"] = gdelt_sentiment["sent_std"].fillna(0)

gdelt_sentiment = gdelt_sentiment.drop_duplicates(
    subset=["ticker", "date", "source"]
)

gdelt_sentiment = gdelt_sentiment.sort_values(["ticker", "date"]).reset_index(drop=True)

Computes a quick summary table showing, for each ticker, the earliest date, latest date and total number of sentiment records in the final GDELT dataset.

In [ ]:
gdelt_sentiment.groupby("ticker")["date"].agg(["min", "max", "count"])

Saves the final GDELT sentiment DataFrame to CSV and downloads it to your local machine from Colab.

In [ ]:
gdelt_sentiment.to_csv("final_gdelt_sentiment_2015_2025.csv", index=False)

from google.colab import files
files.download("final_gdelt_sentiment_2015_2025.csv")

## Exploratory Data Analysis

Loads each ticker’s CSV file, converts the Date column to datetime, sorts the rows by date, and stores the cleaned DataFrame in a dictionary for later use.

In [ ]:
import pandas as pd
import os

tickers = ["WMT", "NVDA", "AMD", "KO", "JNJ", "MSFT", "AAPL", "AMZN", "NFLX"]

data = {}

for ticker in tickers:
    file = f"{ticker}_raw_2005_2025.csv"

    df = pd.read_csv(file)

    # Standardise
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    data[ticker] = df

**Creates a quick EDA summary for each ticker by extracting date range, row count, missing values, and basic Close‑price statistics, then stores everything in a summary DataFrame.**

In [ ]:
eda_summary = []

for ticker, df in data.items():
    eda_summary.append({
        "ticker": ticker,
        "start_date": df["Date"].min(),
        "end_date": df["Date"].max(),
        "rows": len(df),
        "missing_values": df.isna().sum().sum(),
        "min_close": df["Close"].min(),
        "max_close": df["Close"].max(),
        "mean_close": df["Close"].mean(),
        "std_close": df["Close"].std()
    })

eda_summary_df = pd.DataFrame(eda_summary)
eda_summary_df

Plots each ticker’s Close price from 2005–2025 in a multi‑subplot grid, arranging the charts neatly and removing any unused axes

In [ ]:
import matplotlib.pyplot as plt
import math

tickers = list(data.keys())
n = len(tickers)

cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows*4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    df = data[ticker]

    axes[i].plot(df["Date"], df["Close"])
    axes[i].set_title(f"{ticker} (2005–2025)")
    axes[i].set_xlabel("Date")
    axes[i].set_ylabel("Price")

# remove empty plots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Plots each ticker’s yearly average Close price in a multi‑subplot grid, using one subplot per ticker and removing any unused rows.

In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(18, rows*4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    df = data[ticker]

    yearly = df.groupby(df["Date"].dt.year)["Close"].mean()

    axes[i].plot(yearly.index, yearly.values)
    axes[i].set_title(f"{ticker} Yearly Avg Price")
    axes[i].set_xlabel("Year")
    axes[i].set_ylabel("Avg Price")

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Plots each year (2005–2025) in its own subplot, showing all tickers’ daily Close prices for that year.

In [ ]:
import matplotlib.pyplot as plt
import math

years = range(2005, 2026)
tickers = list(data.keys())

cols = 3
rows = math.ceil(len(years) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows*4))
axes = axes.flatten()

for i, year in enumerate(years):

    ax = axes[i]

    for ticker in tickers:
        df = data[ticker]

        temp = df[df["Date"].dt.year == year]

        if not temp.empty:
            ax.plot(temp["Date"], temp["Close"], label=ticker, alpha=0.7)

    ax.set_title(f"Year {year}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")

# remove empty plots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

For each stock, it creates a grid of plots showing how the price moved in every year from 2005 to 2025.

In [ ]:
import matplotlib.pyplot as plt
import math

years = list(range(2005, 2026))
tickers = list(data.keys())

cols = 4   # 👈 change to 3 if you prefer

for ticker in tickers:

    df = data[ticker].copy()

    print(f"\n===== {ticker} =====")

    rows = math.ceil(len(years) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 3))
    axes = axes.flatten()

    for i, year in enumerate(years):

        ax = axes[i]

        temp = df[df["Date"].dt.year == year].copy()

        if not temp.empty:
            ax.plot(temp["Date"], temp["Close"])

        ax.set_title(str(year))
        ax.set_xlabel("")
        ax.set_ylabel("")

    # remove empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"{ticker} Year-by-Year Price Movement", fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

For every stock, it shows how the price changed each year after normalising the first value to 1, arranged neatly in a grid

In [ ]:
for ticker in tickers:

    df = data[ticker].copy()

    print(f"\n===== {ticker} =====")

    rows = math.ceil(len(years) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 3))
    axes = axes.flatten()

    for i, year in enumerate(years):

        ax = axes[i]

        temp = df[df["Date"].dt.year == year].copy()

        if not temp.empty:
            temp["norm"] = temp["Close"] / temp["Close"].iloc[0]
            ax.plot(temp["Date"], temp["norm"])

        ax.set_title(str(year))

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"{ticker} Year-by-Year Normalised Movement", fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

For each stock and each year, it creates a 12‑panel chart showing how the price moved month by month.

In [ ]:
import matplotlib.pyplot as plt
import math

tickers = list(data.keys())
years = range(2005, 2026)

month_names = [
    "Jan","Feb","Mar","Apr","May","Jun",
    "Jul","Aug","Sep","Oct","Nov","Dec"
]

cols = 4
rows = 3  # 12 months

for ticker in tickers:

    df = data[ticker].copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day

    print(f"\n===== {ticker} =====")

    for year in years:

        df_year = df[df["Year"] == year]

        if df_year.empty:
            continue

        fig, axes = plt.subplots(rows, cols, figsize=(16, 10))
        axes = axes.flatten()

        for m in range(1, 13):

            ax = axes[m-1]
            temp = df_year[df_year["Month"] == m]

            if not temp.empty:
                ax.plot(temp["Day"], temp["Close"])

            ax.set_title(month_names[m-1])
            ax.set_xlabel("Day")
            ax.set_ylabel("Price")

        plt.suptitle(f"{ticker} - {year} Monthly Breakdown", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

For each stock and each year, it creates a 12‑panel chart where every month’s price is normalised to start at 1

In [ ]:
for ticker in tickers:

    df = data[ticker].copy()
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day

    print(f"\n===== {ticker} =====")

    for year in years:

        df_year = df[df["Year"] == year]

        if df_year.empty:
            continue

        fig, axes = plt.subplots(3, 4, figsize=(16, 10))
        axes = axes.flatten()

        for m in range(1, 13):

            ax = axes[m-1]
            temp = df_year[df_year["Month"] == m].copy()

            if not temp.empty:
                temp["norm"] = temp["Close"] / temp["Close"].iloc[0]
                ax.plot(temp["Day"], temp["norm"])

            ax.set_title(month_names[m-1])
            ax.set_xlabel("Day")
            ax.set_ylabel("Norm")

        plt.suptitle(f"{ticker} - {year} Normalised Monthly Movement", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

Adds a log_return column to each ticker’s DataFrame by calculating the log‑change in price from one day to the next.

In [ ]:
for ticker in data:
    data[ticker] = data[ticker].copy()
    data[ticker]["log_return"] = np.log(
        data[ticker]["Close"] / data[ticker]["Close"].shift(1)
    )

data["AAPL"].head()

Runs an ADF test on both price levels and log returns for every ticker, then stores the test statistics, p‑values, and stationarity flags in a summary DataFrame.

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_results = []

for ticker in data:
    df = data[ticker].copy()

    price_series = df["Close"].dropna()
    return_series = df["log_return"].dropna()

    price_adf = adfuller(price_series)
    return_adf = adfuller(return_series)

    adf_results.append({
        "ticker": ticker,
        "price_adf_stat": price_adf[0],
        "price_p_value": price_adf[1],
        "price_stationary": price_adf[1] < 0.05,
        "return_adf_stat": return_adf[0],
        "return_p_value": return_adf[1],
        "return_stationary": return_adf[1] < 0.05
    })

adf_results_df = pd.DataFrame(adf_results)
adf_results_df

Normalises each month’s Close prices for every ticker and year, then plots all 12 months in a 3×4 grid.

In [ ]:
cols = 4
rows = math.ceil(len(tickers) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    returns = data[ticker]["log_return"].dropna()

    axes[i].hist(returns, bins=100)
    axes[i].set_title(f"{ticker} Log Return Distribution")
    axes[i].set_xlabel("Log Return")
    axes[i].set_ylabel("Frequency")
    axes[i].grid(True, linestyle="--", alpha=0.4)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Computes a 30‑day rolling volatility from log returns for each ticker and plots the volatility series in a multi‑subplot grid.

In [ ]:
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    df = data[ticker].copy()
    df["rolling_volatility_30d"] = df["log_return"].rolling(30).std()

    axes[i].plot(df["Date"], df["rolling_volatility_30d"])
    axes[i].set_title(f"{ticker} 30-Day Rolling Volatility")
    axes[i].set_xlabel("Date")
    axes[i].set_ylabel("Volatility")
    axes[i].grid(True, linestyle="--", alpha=0.4)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Plots the autocorrelation function (ACF) of log returns for each ticker using 50 lags, arranged in a multi‑subplot grid.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    returns = data[ticker]["log_return"].dropna()
    plot_acf(returns, lags=50, ax=axes[i])
    axes[i].set_title(f"{ticker} ACF of Log Returns")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Combines all tickers’ log‑return series into one DataFrame aligned by date, computes their correlation matrix, and visualises it using a heatmap.

In [ ]:
import seaborn as sns

returns_df = pd.DataFrame()

for ticker in tickers:
    temp = data[ticker][["Date", "log_return"]].copy()
    temp = temp.rename(columns={"log_return": ticker})

    if returns_df.empty:
        returns_df = temp
    else:
        returns_df = returns_df.merge(temp, on="Date", how="outer")

returns_df = returns_df.sort_values("Date").set_index("Date")

plt.figure(figsize=(10, 7))
sns.heatmap(returns_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Log Returns")
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

Performs seasonal decomposition on each ticker’s log‑return series for two time periods (2005–2015 and 2015–2025), then plots the observed, trend, seasonal, and residual components.

In [ ]:
periods = [
    ("2005", "2015"),
    ("2015", "2025")
]

for ticker in tickers:

    df = data[ticker].copy()
    df = df.set_index("Date")

    df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))
    df = df.dropna()

    for start, end in periods:

        temp = df[start:end]

        if len(temp) < 300:
            continue

        result = seasonal_decompose(
            temp["log_return"],
            model="additive",
            period=252
        )

        fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

        axes[0].plot(result.observed)
        axes[0].set_title("Observed")

        axes[1].plot(result.trend)
        axes[1].set_title("Trend")

        axes[2].plot(result.seasonal)
        axes[2].set_title("Seasonal")

        axes[3].plot(result.resid)
        axes[3].set_title("Residual")

        plt.suptitle(f"{ticker} Decomposition of Returns ({start}–{end})", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

## Technical Analysis & Feature Engineering

Creates a full technical‑indicator feature set for each ticker, including log returns, lagged returns, moving averages, EMAs, momentum, volatility, price–MA ratios, volume change, and the next‑day return target.

In [ ]:
import numpy as np
import pandas as pd

def add_technical_features(df):
    df = df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    # Log return
    df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))

    # Lagged returns
    df["return_lag1"] = df["log_return"].shift(1)
    df["return_lag2"] = df["log_return"].shift(2)
    df["return_lag5"] = df["log_return"].shift(5)

    # Moving averages
    df["ma_7"] = df["Close"].rolling(7).mean()
    df["ma_30"] = df["Close"].rolling(30).mean()

    # Exponential moving averages
    df["ema_7"] = df["Close"].ewm(span=7, adjust=False).mean()
    df["ema_30"] = df["Close"].ewm(span=30, adjust=False).mean()

    # Momentum
    df["momentum_7"] = df["Close"] - df["Close"].shift(7)
    df["momentum_30"] = df["Close"] - df["Close"].shift(30)

    # Rolling volatility
    df["volatility_7"] = df["log_return"].rolling(7).std()
    df["volatility_30"] = df["log_return"].rolling(30).std()

    # Price relative to moving average
    df["close_ma7_ratio"] = df["Close"] / df["ma_7"]
    df["close_ma30_ratio"] = df["Close"] / df["ma_30"]

    # Volume feature
    df["volume_change"] = df["Volume"].pct_change()

    # Prediction target: tomorrow's return
    df["target_next_return"] = df["log_return"].shift(-1)

    return df

In [ ]:
featured_data = {}

for ticker in data:
    featured_data[ticker] = add_technical_features(data[ticker])

featured_data["AAPL"].head()

Builds a summary table for each ticker showing date range, row/column counts, and total missing values after feature engineering.

In [ ]:
feature_check = []

for ticker, df in featured_data.items():
    feature_check.append({
        "ticker": ticker,
        "start_date": df["Date"].min(),
        "end_date": df["Date"].max(),
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": df.isna().sum().sum()
    })

feature_check_df = pd.DataFrame(feature_check)
feature_check_df

Loops through all feature‑engineered DataFrames, extracts basic metadata (start/end dates, shape, missing‑value count), and stores everything in a clean summary DataFrame.

In [ ]:
feature_cols = [
    "log_return",
    "return_lag1",
    "return_lag2",
    "return_lag5",
    "ma_7",
    "ma_30",
    "ema_7",
    "ema_30",
    "momentum_7",
    "momentum_30",
    "volatility_7",
    "volatility_30",
    "close_ma7_ratio",
    "close_ma30_ratio",
    "volume_change"
]

target_col = "target_next_return"

Replaces infinite values with NaN, drops all rows containing missing values, and resets the index for every ticker’s feature‑engineered dataset.

In [ ]:
for ticker in featured_data:
    featured_data[ticker] = (
        featured_data[ticker]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .reset_index(drop=True)
    )

Builds a summary table for each ticker showing the cleaned dataset’s date range, number of rows/columns, and remaining missing values after feature engineering.

In [ ]:
feature_check = []

for ticker, df in featured_data.items():
    feature_check.append({
        "ticker": ticker,
        "start_date": df["Date"].min(),
        "end_date": df["Date"].max(),
        "rows": len(df),
        "columns": len(df.columns),
        "missing_values": df.isna().sum().sum()
    })

feature_check_df = pd.DataFrame(feature_check)
feature_check_df

## Split Dataset

Builds a summary table for each ticker showing the cleaned dataset’s date range, number of rows/columns, and remaining missing values after feature engineering.

In [ ]:
splits = {}

for ticker in featured_data:
    df = featured_data[ticker].copy()
    df = df.sort_values("Date").reset_index(drop=True)

    train = df[df["Date"] < "2019-01-01"]
    val = df[(df["Date"] >= "2019-01-01") & (df["Date"] < "2022-01-01")]
    test = df[df["Date"] >= "2022-01-01"]

    splits[ticker] = {
        "train": train,
        "val": val,
        "test": test
    }

Shows how many rows and columns ended up in the train/val/test sets for every stock.

In [ ]:
for ticker in splits:
    print(f"\n{ticker}")
    print("Train:", splits[ticker]["train"].shape)
    print("Val:  ", splits[ticker]["val"].shape)
    print("Test: ", splits[ticker]["test"].shape)

Verifies that each ticker’s train, validation, and test sets have strictly non‑overlapping date ranges and contain no duplicate dates.

In [ ]:
for ticker in splits:
    train = splits[ticker]["train"]
    val = splits[ticker]["val"]
    test = splits[ticker]["test"]

    assert train["Date"].max() < val["Date"].min(), f"{ticker}: Train overlaps validation"
    assert val["Date"].max() < test["Date"].min(), f"{ticker}: Validation overlaps test"

    assert train["Date"].is_unique, f"{ticker}: Duplicate dates in train"
    assert val["Date"].is_unique, f"{ticker}: Duplicate dates in validation"
    assert test["Date"].is_unique, f"{ticker}: Duplicate dates in test"

print("All checks passed. No overlap or duplicate dates.")

## Linear Regression Baseline: Actual vs Predicted Test Returns

Imports the Linear Regression model, error metrics (MAE and RMSE), and core libraries (NumPy, pandas, Matplotlib) needed for model training and evaluation.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Evaluates predictions by computing absolute error, squared‑error‑based RMSE, average bias (mean error), and the percentage of times the model predicted the correct sign of return.

In [ ]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mean_error = np.mean(y_pred - y_true)
    directional_accuracy = np.mean(np.sign(y_true) == np.sign(y_pred))

    return {
        "MAE": mae,
        "RMSE": rmse,
        "Mean_Error": mean_error,
        "Directional_Accuracy": directional_accuracy
    }

Checks every ticker’s train/val/test split for any remaining infinity values in the feature columns and prints the counts if found.

In [ ]:
for ticker in splits:
    for split_name in ["train", "val", "test"]:
        df = splits[ticker][split_name]

        inf_counts = np.isinf(df[feature_cols]).sum()

        if inf_counts.sum() > 0:
            print(f"\n{ticker} - {split_name}")
            print(inf_counts[inf_counts > 0])

Goes through every split, removes infinities, drops rows with missing features or target, and updates the split with the cleaned version.

In [ ]:
for ticker in splits:
    for split_name in ["train", "val", "test"]:

        df = splits[ticker][split_name].copy()

        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

        splits[ticker][split_name] = df

Prints NaN and Inf counts for every split of every ticker.

In [ ]:
for ticker in splits:
    for split_name in ["train", "val", "test"]:

        df = splits[ticker][split_name]

        nan_count = df[feature_cols + [target_col]].isna().sum().sum()
        inf_count = np.isinf(df[feature_cols + [target_col]]).sum().sum()

        print(f"{ticker} {split_name} | NaN: {nan_count} | Inf: {inf_count}")

Runs two baselines (naive and linear regression) for each ticker and saves their validation/test metrics.

In [ ]:
baseline_results = []

for ticker in splits:

    train = splits[ticker]["train"].copy()
    val = splits[ticker]["val"].copy()
    test = splits[ticker]["test"].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_val = val[feature_cols]
    y_val = val[target_col]

    X_test = test[feature_cols]
    y_test = test[target_col]

    # Naive baseline: tomorrow's return = today's return
    naive_val_pred = val["log_return"].values
    naive_test_pred = test["log_return"].values

    naive_val_metrics = evaluate_model(y_val, naive_val_pred)
    naive_test_metrics = evaluate_model(y_test, naive_test_pred)

    # Linear Regression baseline: fit on TRAIN only
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    lr_val_pred = lr_model.predict(X_val)
    lr_test_pred = lr_model.predict(X_test)

    lr_val_metrics = evaluate_model(y_val, lr_val_pred)
    lr_test_metrics = evaluate_model(y_test, lr_test_pred)

    baseline_results.append({
        "ticker": ticker,

        "naive_val_MAE": naive_val_metrics["MAE"],
        "naive_val_RMSE": naive_val_metrics["RMSE"],
        "naive_val_Mean_Error": naive_val_metrics["Mean_Error"],
        "naive_val_Directional_Accuracy": naive_val_metrics["Directional_Accuracy"],

        "naive_test_MAE": naive_test_metrics["MAE"],
        "naive_test_RMSE": naive_test_metrics["RMSE"],
        "naive_test_Mean_Error": naive_test_metrics["Mean_Error"],
        "naive_test_Directional_Accuracy": naive_test_metrics["Directional_Accuracy"],

        "lr_val_MAE": lr_val_metrics["MAE"],
        "lr_val_RMSE": lr_val_metrics["RMSE"],
        "lr_val_Mean_Error": lr_val_metrics["Mean_Error"],
        "lr_val_Directional_Accuracy": lr_val_metrics["Directional_Accuracy"],

        "lr_test_MAE": lr_test_metrics["MAE"],
        "lr_test_RMSE": lr_test_metrics["RMSE"],
        "lr_test_Mean_Error": lr_test_metrics["Mean_Error"],
        "lr_test_Directional_Accuracy": lr_test_metrics["Directional_Accuracy"]
    })

baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

Creates a DataFrame that compares naïve and linear‑regression test metrics and calculates how much better or worse the regression model performs.

In [ ]:
comparison_df = baseline_results_df[[
    "ticker",
    "naive_test_MAE",
    "lr_test_MAE",
    "naive_test_RMSE",
    "lr_test_RMSE",
    "naive_test_Directional_Accuracy",
    "lr_test_Directional_Accuracy"
]].copy()

comparison_df["MAE_improvement"] = comparison_df["naive_test_MAE"] - comparison_df["lr_test_MAE"]
comparison_df["RMSE_improvement"] = comparison_df["naive_test_RMSE"] - comparison_df["lr_test_RMSE"]

comparison_df["Direction_improvement"] = (
    comparison_df["lr_test_Directional_Accuracy"] -
    comparison_df["naive_test_Directional_Accuracy"]
)

comparison_df

Trains a Linear Regression model for each ticker using the training split, predicts next‑day returns on the test split, and plots actual vs. predicted returns

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression

tickers = list(splits.keys())

cols = 3
rows = math.ceil(len(tickers) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_test = test[feature_cols]
    y_test = test[target_col]

    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    lr_test_pred = lr_model.predict(X_test)

    ax = axes[i]

    ax.plot(test["Date"], y_test.values, label="Actual", linewidth=1)
    ax.plot(test["Date"], lr_test_pred, label="Predicted", linewidth=1)

    ax.set_title(f"{ticker}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Next-Day Return")
    ax.grid(True, linestyle="--", alpha=0.4)

    # Better date formatting
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.tick_params(axis="x", rotation=45)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)

plt.suptitle("Linear Regression Baseline: Actual vs Predicted Test Returns", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

Fits a regression model for each stock, predicts test returns, evaluates performance separately for each year, and collects the metrics.

In [ ]:
yearly_lr_results = []

for ticker in splits:
    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_test = test[feature_cols]
    y_test = test[target_col]

    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    test = test.copy()
    test["prediction"] = lr_model.predict(X_test)
    test["year"] = test["Date"].dt.year

    for year, group in test.groupby("year"):
        metrics = evaluate_model(group[target_col], group["prediction"])

        yearly_lr_results.append({
            "ticker": ticker,
            "year": year,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "Directional_Accuracy": metrics["Directional_Accuracy"]
        })

yearly_lr_results_df = pd.DataFrame(yearly_lr_results)
yearly_lr_results_df

##GBM (Geometric Brownian Motion) Model

Checks if important variables and functions are loaded in memory and prints True/False for each.

In [ ]:
print("splits exists:", "splits" in globals())
print("simulate_gbm exists:", "simulate_gbm" in globals())
print("tickers exists:", "tickers" in globals())
print("mean_absolute_error exists:", "mean_absolute_error" in globals())
print("mean_squared_error exists:", "mean_squared_error" in globals())

Loads the core libraries for data handling, math operations, plotting, and model‑error calculations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from sklearn.metrics import mean_absolute_error, mean_squared_error

Shows the start/end dates for each split and checks that train, val, and test don’t overlap and have no duplicate dates.

In [ ]:
for ticker in splits:
    train = splits[ticker]["train"]
    val = splits[ticker]["val"]
    test = splits[ticker]["test"]

    print(f"\n{ticker}")
    print("Train:", train["Date"].min(), "to", train["Date"].max())
    print("Val:  ", val["Date"].min(), "to", val["Date"].max())
    print("Test: ", test["Date"].min(), "to", test["Date"].max())

    assert train["Date"].max() < val["Date"].min(), f"{ticker}: Train overlaps validation"
    assert val["Date"].max() < test["Date"].min(), f"{ticker}: Validation overlaps test"

    assert train["Date"].is_unique, f"{ticker}: Duplicate dates in train"
    assert val["Date"].is_unique, f"{ticker}: Duplicate dates in validation"
    assert test["Date"].is_unique, f"{ticker}: Duplicate dates in test"

print("\nAll leakage checks passed. No overlap or duplicate dates.")

Creates many simulated price paths using the GBM formula. Each simulation starts at S₀ and evolves step‑by‑step using drift, volatility, and random shocks.

In [ ]:
np.random.seed(42)

def simulate_gbm(S0, mu, sigma, steps, n_sims=1000):
    dt = 1 / 252
    simulations = np.zeros((steps, n_sims))

    for sim in range(n_sims):
        prices = [S0]

        for t in range(1, steps):
            Z = np.random.normal()
            St = prices[-1] * np.exp(
                (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
            )
            prices.append(St)

        simulations[:, sim] = prices

    return simulations

**GBM Basic Models**

For each ticker, estimates μ and σ from the training set, simulates 1,000 GBM price paths across the full test period, computes the mean and 5–95% uncertainty band, evaluates MAE/RMSE against actual prices, stores the results, and plots actual vs. GBM mean

In [ ]:
gbm_full_results = []
gbm_full_predictions = {}

tickers = list(splits.keys())

cols = 3
rows = math.ceil(len(tickers) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):

    ax = axes[i]

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    # Parameters estimated from TRAIN only
    mu = train["log_return"].mean() * 252
    sigma = train["log_return"].std() * np.sqrt(252)

    # S0 = first price of full test period
    S0 = test["Close"].iloc[0]
    steps = len(test)

    sims = simulate_gbm(S0, mu, sigma, steps, n_sims=1000)

    gbm_mean = sims.mean(axis=1)
    lower = np.percentile(sims, 5, axis=1)
    upper = np.percentile(sims, 95, axis=1)

    actual = test["Close"].values

    mae = mean_absolute_error(actual, gbm_mean)
    rmse = np.sqrt(mean_squared_error(actual, gbm_mean))

    gbm_full_results.append({
        "ticker": ticker,
        "gbm_full_mae": mae,
        "gbm_full_rmse": rmse
    })

    gbm_full_predictions[ticker] = pd.DataFrame({
        "Date": test["Date"].values,
        "actual": actual,
        "gbm_mean": gbm_mean,
        "lower_5": lower,
        "upper_95": upper
    })

    ax.plot(test["Date"], actual, color="black", linewidth=2, label="Actual")
    ax.plot(test["Date"], gbm_mean, color="blue", linewidth=1.5, label="GBM Mean")
    ax.fill_between(test["Date"], lower, upper, color="blue", alpha=0.15, label="5%-95% Band")

    ax.set_title(f"{ticker}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")
    ax.grid(True, linestyle="--", alpha=0.4)

    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(1)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)

plt.suptitle("Basic GBM: Full Test-Period Forecast vs Actual", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

gbm_full_results_df = pd.DataFrame(gbm_full_results)
gbm_full_results_df

Displays the summary table containing each ticker’s full‑period GBM MAE and RMSE, computed using the mean of 1,000 simulated GBM paths.

In [ ]:
gbm_full_results_df

For every stock, the code takes drift and volatility from the training period, slices the test data into individual years, simulates 1,000 GBM paths for each slice, extracts the mean and confidence band, evaluates MAE/RMSE, stores the metrics, and produces a clean multi‑panel figure showing actual vs. GBM forecasts year‑by‑year.

In [ ]:
import matplotlib.dates as mdates

years = sorted(splits[tickers[0]]["test"]["Date"].dt.year.unique())

gbm_yearly_results = []

cols = 2
rows = math.ceil(len(years) / cols)

for ticker in tickers:

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    mu = train["log_return"].mean() * 252
    sigma = train["log_return"].std() * np.sqrt(252)

    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4.8))
    axes = axes.flatten()

    for i, year in enumerate(years):

        ax = axes[i]
        test_year = test[test["Date"].dt.year == year].copy()

        if test_year.empty:
            ax.set_title(f"{year} — No Data")
            ax.axis("off")
            continue

        # S0 = first price of this year
        S0 = test_year["Close"].iloc[0]
        steps = len(test_year)

        sims = simulate_gbm(S0, mu, sigma, steps, n_sims=1000)

        gbm_mean = sims.mean(axis=1)
        lower = np.percentile(sims, 5, axis=1)
        upper = np.percentile(sims, 95, axis=1)

        actual = test_year["Close"].values

        mae = mean_absolute_error(actual, gbm_mean)
        rmse = np.sqrt(mean_squared_error(actual, gbm_mean))

        gbm_yearly_results.append({
            "ticker": ticker,
            "year": year,
            "gbm_yearly_mae": mae,
            "gbm_yearly_rmse": rmse
        })

        ax.plot(
            test_year["Date"],
            actual,
            color="black",
            linewidth=2,
            label="Actual"
        )

        ax.plot(
            test_year["Date"],
            gbm_mean,
            color="blue",
            linewidth=1.5,
            label="GBM Mean"
        )

        ax.fill_between(
            test_year["Date"],
            lower,
            upper,
            color="blue",
            alpha=0.15,
            label="5%-95% Band"
        )

        ax.set_title(f"{ticker} — {year}", fontsize=11, pad=10)
        ax.set_xlabel("Date")
        ax.set_ylabel("Price")
        ax.grid(True, linestyle="--", alpha=0.4)

        # Better date formatting
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        ax.tick_params(axis="x", rotation=45)

        # Clear subplot borders
        for spine in ax.spines.values():
            spine.set_edgecolor("black")
            spine.set_linewidth(1.1)

    # Remove empty axes
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # Main title
    fig.suptitle(
        f"{ticker}: Basic GBM Year-by-Year Forecast vs Actual",
        fontsize=18,
        fontweight="bold",
        y=0.98
    )

    # One shared legend, placed below title
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.94),
        ncol=3,
        frameon=True
    )

    # Proper spacing
    plt.subplots_adjust(
        top=0.86,
        hspace=0.45,
        wspace=0.25
    )

    plt.show()

gbm_yearly_results_df = pd.DataFrame(gbm_yearly_results)
gbm_yearly_results_df.head()

Prints the year‑by‑year GBM performance results.

In [ ]:
gbm_yearly_results_df

Shows each ticker’s average GBM performance and its best and worst years.

In [ ]:
gbm_yearly_summary = gbm_yearly_results_df.groupby("ticker").agg(
    avg_yearly_mae=("gbm_yearly_mae", "mean"),
    avg_yearly_rmse=("gbm_yearly_rmse", "mean"),
    best_yearly_mae=("gbm_yearly_mae", "min"),
    worst_yearly_mae=("gbm_yearly_mae", "max")
).reset_index()

gbm_yearly_summary

For each ticker and each test‑set year, the code simulates 1,000 GBM paths for every month using μ and σ estimated from the training period, computes monthly MAE/RMSE, stores the results, and plots actual vs. GBM mean with a 5–95% uncertainty band in a 3×4 grid.

In [ ]:
import matplotlib.dates as mdates

month_names = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

gbm_monthly_results = []

for ticker in tickers:

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    # Parameters estimated from TRAIN only
    mu = train["log_return"].mean() * 252
    sigma = train["log_return"].std() * np.sqrt(252)

    for year in years:

        test_year = test[test["Date"].dt.year == year].copy()

        if test_year.empty:
            continue

        fig, axes = plt.subplots(3, 4, figsize=(20, 12))
        axes = axes.flatten()

        for month in range(1, 13):

            ax = axes[month - 1]

            test_month = test_year[test_year["Date"].dt.month == month].copy()

            if test_month.empty:
                ax.set_title(f"{month_names[month - 1]} — No Data")
                ax.axis("off")
                continue

            # S0 = first price of this month
            S0 = test_month["Close"].iloc[0]
            steps = len(test_month)

            sims = simulate_gbm(S0, mu, sigma, steps, n_sims=1000)

            gbm_mean = sims.mean(axis=1)
            lower = np.percentile(sims, 5, axis=1)
            upper = np.percentile(sims, 95, axis=1)

            actual = test_month["Close"].values

            mae = mean_absolute_error(actual, gbm_mean)
            rmse = np.sqrt(mean_squared_error(actual, gbm_mean))

            gbm_monthly_results.append({
                "ticker": ticker,
                "year": year,
                "month": month,
                "gbm_monthly_mae": mae,
                "gbm_monthly_rmse": rmse
            })

            ax.plot(
                test_month["Date"].dt.day,
                actual,
                color="black",
                linewidth=2,
                label="Actual"
            )

            ax.plot(
                test_month["Date"].dt.day,
                gbm_mean,
                color="blue",
                linewidth=1.5,
                label="GBM Mean"
            )

            ax.fill_between(
                test_month["Date"].dt.day,
                lower,
                upper,
                color="blue",
                alpha=0.15,
                label="5%-95% Band"
            )

            ax.set_title(month_names[month - 1], fontsize=11, pad=8)
            ax.set_xlabel("Day")
            ax.set_ylabel("Price")
            ax.grid(True, linestyle="--", alpha=0.4)

            # Clear subplot borders
            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1.1)

        # Main title
        fig.suptitle(
            f"{ticker}: Basic GBM Month-by-Month Forecast vs Actual ({year})",
            fontsize=18,
            fontweight="bold",
            y=0.985
        )

        # Shared legend
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.94),
            ncol=3,
            frameon=True
        )

        # Spacing
        plt.subplots_adjust(
            top=0.88,
            hspace=0.50,
            wspace=0.30
        )

        plt.show()

gbm_monthly_results_df = pd.DataFrame(gbm_monthly_results)
gbm_monthly_results_df.head()

Summarises the monthly GBM errors for each stock by showing the average MAE/RMSE and the best/worst monthly MAE.

In [ ]:
gbm_monthly_summary = gbm_monthly_results_df.groupby("ticker").agg(
    avg_monthly_mae=("gbm_monthly_mae", "mean"),
    avg_monthly_rmse=("gbm_monthly_rmse", "mean"),
    best_monthly_mae=("gbm_monthly_mae", "min"),
    worst_monthly_mae=("gbm_monthly_mae", "max")
).reset_index()

gbm_monthly_summary

Summarises each stock’s yearly GBM performance by showing the average MAE/RMSE and the best and worst yearly MAE.

In [ ]:
gbm_yearly_summary = gbm_yearly_results_df.groupby("ticker").agg(
    avg_yearly_mae=("gbm_yearly_mae", "mean"),
    avg_yearly_rmse=("gbm_yearly_rmse", "mean"),
    best_yearly_mae=("gbm_yearly_mae", "min"),
    worst_yearly_mae=("gbm_yearly_mae", "max")
).reset_index()

gbm_yearly_summary

**Periodic-Correction GBM**

For each ticker and each test‑set month, the code uses the previous 60 trading days (strictly before the forecast month) to re‑estimate μ and σ, simulates 1,000 GBM paths for that month, computes MAE/RMSE, stores the results, and plots actual vs. periodic‑GBM mean with a 5–95% uncertainty band

In [ ]:
gbm_periodic_monthly_results = []

window_size = 60  # previous 60 trading days

for ticker in tickers:

    # Use full featured data so previous history before test month is available
    df = featured_data[ticker].copy()
    df = df.sort_values("Date").reset_index(drop=True)

    test = splits[ticker]["test"].copy()
    test_years = sorted(test["Date"].dt.year.unique())

    for year in test_years:

        test_year = test[test["Date"].dt.year == year].copy()

        if test_year.empty:
            continue

        y_min = test_year["Close"].min() * 0.95
        y_max = test_year["Close"].max() * 1.05

        fig, axes = plt.subplots(4, 3, figsize=(18, 16))
        axes = axes.flatten()

        for month in range(1, 13):

            ax = axes[month - 1]

            test_month = test_year[test_year["Date"].dt.month == month].copy()

            if test_month.empty:
                ax.set_title(f"{month_names[month - 1]} — No Data")
                ax.axis("off")
                continue

            forecast_start_date = test_month["Date"].iloc[0]

            # Use only data BEFORE the current forecast month
            historical_window = df[df["Date"] < forecast_start_date].tail(window_size)

            if len(historical_window) < window_size:
                ax.set_title(f"{month_names[month - 1]} — Not Enough History")
                ax.axis("off")
                continue

            # Estimate parameters from previous history only
            mu = historical_window["log_return"].mean() * 252
            sigma = historical_window["log_return"].std() * np.sqrt(252)

            # S0 = first price of current month
            S0 = test_month["Close"].iloc[0]
            steps = len(test_month)

            sims = simulate_gbm(S0, mu, sigma, steps, n_sims=1000)

            gbm_mean = sims.mean(axis=1)
            lower = np.percentile(sims, 5, axis=1)
            upper = np.percentile(sims, 95, axis=1)

            actual = test_month["Close"].values

            mae = mean_absolute_error(actual, gbm_mean)
            rmse = np.sqrt(mean_squared_error(actual, gbm_mean))

            gbm_periodic_monthly_results.append({
                "ticker": ticker,
                "year": year,
                "month": month,
                "window_size": window_size,
                "periodic_monthly_mae": mae,
                "periodic_monthly_rmse": rmse,
                "mu": mu,
                "sigma": sigma
            })

            ax.plot(
                test_month["Date"].dt.day,
                actual,
                color="black",
                linewidth=2,
                label="Actual"
            )

            ax.plot(
                test_month["Date"].dt.day,
                gbm_mean,
                color="green",
                linewidth=1.5,
                label="Periodic GBM Mean"
            )

            ax.fill_between(
                test_month["Date"].dt.day,
                lower,
                upper,
                color="green",
                alpha=0.15,
                label="5%-95% Band"
            )

            ax.set_title(month_names[month - 1], fontsize=11, pad=8)
            ax.set_xlabel("Day")
            ax.set_ylabel("Price")
            ax.set_ylim(y_min, y_max)
            ax.grid(True, linestyle="--", alpha=0.4)

            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1.1)

        fig.suptitle(
            f"{ticker}: Periodic-Correction GBM Month-by-Month Forecast ({year})",
            fontsize=18,
            fontweight="bold",
            y=0.98
        )

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.945),
            ncol=3,
            frameon=True
        )

        plt.subplots_adjust(
            top=0.90,
            hspace=0.45,
            wspace=0.25
        )

        plt.show()

gbm_periodic_monthly_results_df = pd.DataFrame(gbm_periodic_monthly_results)
gbm_periodic_monthly_results_df.head()

Summarises each stock’s periodic‑GBM monthly performance: average errors, best/worst months, and the average drift/volatility used across all rolling windows

In [ ]:
gbm_periodic_monthly_summary = gbm_periodic_monthly_results_df.groupby("ticker").agg(
    avg_periodic_monthly_mae=("periodic_monthly_mae", "mean"),
    avg_periodic_monthly_rmse=("periodic_monthly_rmse", "mean"),
    best_periodic_monthly_mae=("periodic_monthly_mae", "min"),
    worst_periodic_monthly_mae=("periodic_monthly_mae", "max"),
    avg_mu=("mu", "mean"),
    avg_sigma=("sigma", "mean")
).reset_index()

gbm_periodic_monthly_summary

Comparison between Basic GBM vs Periodic GBM

Shows how periodic‑GBM compares to basic GBM by computing the difference in MAE/RMSE and indicating if the periodic model improved.

In [ ]:
gbm_comparison = gbm_monthly_summary.merge(
    gbm_periodic_monthly_summary,
    on="ticker",
    how="inner"
)

gbm_comparison["mae_change"] = (
    gbm_comparison["avg_monthly_mae"] -
    gbm_comparison["avg_periodic_monthly_mae"]
)

gbm_comparison["rmse_change"] = (
    gbm_comparison["avg_monthly_rmse"] -
    gbm_comparison["avg_periodic_monthly_rmse"]
)

gbm_comparison["mae_improved"] = gbm_comparison["mae_change"] > 0
gbm_comparison["rmse_improved"] = gbm_comparison["rmse_change"] > 0

gbm_comparison[[
    "ticker",
    "avg_monthly_mae",
    "avg_periodic_monthly_mae",
    "mae_change",
    "mae_improved",
    "avg_monthly_rmse",
    "avg_periodic_monthly_rmse",
    "rmse_change",
    "rmse_improved"
]]

Checks how GBM performs when using 30‑, 60‑, 120‑, and 252‑day lookback windows, recording the monthly MAE/RMSE for each case.

In [ ]:
window_sizes = [30, 60, 120, 252]

periodic_window_results = []

for window_size in window_sizes:

    for ticker in tickers:

        df = featured_data[ticker].copy()
        df = df.sort_values("Date").reset_index(drop=True)

        test = splits[ticker]["test"].copy()
        test_years = sorted(test["Date"].dt.year.unique())

        for year in test_years:

            test_year = test[test["Date"].dt.year == year].copy()

            for month in range(1, 13):

                test_month = test_year[test_year["Date"].dt.month == month].copy()

                if test_month.empty:
                    continue

                forecast_start_date = test_month["Date"].iloc[0]

                historical_window = df[df["Date"] < forecast_start_date].tail(window_size)

                if len(historical_window) < window_size:
                    continue

                mu = historical_window["log_return"].mean() * 252
                sigma = historical_window["log_return"].std() * np.sqrt(252)

                S0 = test_month["Close"].iloc[0]
                steps = len(test_month)

                sims = simulate_gbm(S0, mu, sigma, steps, n_sims=1000)

                gbm_mean = sims.mean(axis=1)
                actual = test_month["Close"].values

                mae = mean_absolute_error(actual, gbm_mean)
                rmse = np.sqrt(mean_squared_error(actual, gbm_mean))

                periodic_window_results.append({
                    "ticker": ticker,
                    "year": year,
                    "month": month,
                    "window_size": window_size,
                    "mae": mae,
                    "rmse": rmse,
                    "mu": mu,
                    "sigma": sigma
                })

periodic_window_results_df = pd.DataFrame(periodic_window_results)
periodic_window_results_df.head()

Summarises how each window size (30, 60, 120, 252 days) performed for every stock by showing the average MAE/RMSE and the typical drift/volatility values.

In [ ]:
window_summary = periodic_window_results_df.groupby(["ticker", "window_size"]).agg(
    avg_mae=("mae", "mean"),
    avg_rmse=("rmse", "mean"),
    avg_mu=("mu", "mean"),
    avg_sigma=("sigma", "mean")
).reset_index()

window_summary

Finds which window size (30, 60, 120, 252 days) gives the lowest average MAE for each stock.

In [ ]:
best_window_by_ticker = (
    window_summary
    .sort_values(["ticker", "avg_mae"])
    .groupby("ticker")
    .first()
    .reset_index()
)

best_window_by_ticker

Compares basic GBM to each ticker’s best periodic‑GBM window, shows how much MAE/RMSE changed, and marks whether the periodic model improved.

In [ ]:
best_periodic_comparison = gbm_monthly_summary.merge(
    best_window_by_ticker,
    on="ticker",
    how="inner"
)

best_periodic_comparison["mae_change"] = (
    best_periodic_comparison["avg_monthly_mae"] -
    best_periodic_comparison["avg_mae"]
)

best_periodic_comparison["rmse_change"] = (
    best_periodic_comparison["avg_monthly_rmse"] -
    best_periodic_comparison["avg_rmse"]
)

best_periodic_comparison["periodic_mae_improved"] = (
    best_periodic_comparison["mae_change"] > 0
)

best_periodic_comparison["periodic_rmse_improved"] = (
    best_periodic_comparison["rmse_change"] > 0
)

best_periodic_comparison[[
    "ticker",
    "avg_monthly_mae",
    "window_size",
    "avg_mae",
    "mae_change",
    "periodic_mae_improved",
    "avg_monthly_rmse",
    "avg_rmse",
    "rmse_change",
    "periodic_rmse_improved"
]]

Saves the table comparing basic GBM to the best periodic‑GBM window for each stock to csv file.

In [ ]:
best_periodic_comparison.to_csv(
    "gbm_basic_vs_periodic_comparison.csv",
    index=False
)

best_periodic_comparison

Creates one final table showing: basic GBM errors (full, yearly, monthly) and the best periodic‑GBM window with its average MAE/RMSE.

In [ ]:
final_gbm_results = gbm_full_results_df.merge(
    gbm_yearly_summary,
    on="ticker",
    how="inner"
).merge(
    gbm_monthly_summary,
    on="ticker",
    how="inner"
).merge(
    best_window_by_ticker,
    on="ticker",
    how="inner"
)

final_gbm_results = final_gbm_results.rename(columns={
    "gbm_full_mae": "basic_full_mae",
    "gbm_full_rmse": "basic_full_rmse",
    "avg_yearly_mae": "basic_yearly_avg_mae",
    "avg_yearly_rmse": "basic_yearly_avg_rmse",
    "avg_monthly_mae": "basic_monthly_avg_mae",
    "avg_monthly_rmse": "basic_monthly_avg_rmse",
    "window_size": "best_periodic_window",
    "avg_mae": "periodic_best_avg_mae",
    "avg_rmse": "periodic_best_avg_rmse"
})

final_gbm_results = final_gbm_results[[
    "ticker",
    "basic_full_mae",
    "basic_full_rmse",
    "basic_yearly_avg_mae",
    "basic_yearly_avg_rmse",
    "basic_monthly_avg_mae",
    "basic_monthly_avg_rmse",
    "best_periodic_window",
    "periodic_best_avg_mae",
    "periodic_best_avg_rmse"
]]

final_gbm_results

Saves the final GBM comparison table (basic GBM vs. best periodic GBM) into a CSV file.

In [ ]:
final_gbm_results.to_csv("final_gbm_results.csv", index=False)

Recalculates GBM parameters every month using the past 252 days, simulates the month, and measures how close the forecast is to the real prices.

In [ ]:
best_window_size = 252

gbm_periodic_252_monthly_results = []

for ticker in tickers:

    df = featured_data[ticker].copy()
    df = df.sort_values("Date").reset_index(drop=True)

    test = splits[ticker]["test"].copy()
    test_years = sorted(test["Date"].dt.year.unique())

    for year in test_years:

        test_year = test[test["Date"].dt.year == year].copy()

        if test_year.empty:
            continue

        y_min = test_year["Close"].min() * 0.95
        y_max = test_year["Close"].max() * 1.05

        fig, axes = plt.subplots(4, 3, figsize=(18, 16))
        axes = axes.flatten()

        for month in range(1, 13):

            ax = axes[month - 1]
            test_month = test_year[test_year["Date"].dt.month == month].copy()

            if test_month.empty:
                ax.set_title(f"{month_names[month - 1]} — No Data")
                ax.axis("off")
                continue

            forecast_start_date = test_month["Date"].iloc[0]

            # Use only previous 252 trading days
            historical_window = df[df["Date"] < forecast_start_date].tail(best_window_size)

            if len(historical_window) < best_window_size:
                ax.set_title(f"{month_names[month - 1]} — Not Enough History")
                ax.axis("off")
                continue

            # Estimate parameters using past data only
            mu = historical_window["log_return"].mean() * 252
            sigma = historical_window["log_return"].std() * np.sqrt(252)

            # S0 = first price of current month
            S0 = test_month["Close"].iloc[0]
            steps = len(test_month)

            sims = simulate_gbm(S0, mu, sigma, steps, n_sims=1000)

            gbm_mean = sims.mean(axis=1)
            lower = np.percentile(sims, 5, axis=1)
            upper = np.percentile(sims, 95, axis=1)

            actual = test_month["Close"].values

            mae = mean_absolute_error(actual, gbm_mean)
            rmse = np.sqrt(mean_squared_error(actual, gbm_mean))

            gbm_periodic_252_monthly_results.append({
                "ticker": ticker,
                "year": year,
                "month": month,
                "window_size": best_window_size,
                "periodic_252_monthly_mae": mae,
                "periodic_252_monthly_rmse": rmse,
                "mu": mu,
                "sigma": sigma
            })

            ax.plot(
                test_month["Date"].dt.day,
                actual,
                color="black",
                linewidth=2,
                label="Actual"
            )

            ax.plot(
                test_month["Date"].dt.day,
                gbm_mean,
                color="green",
                linewidth=1.5,
                label="Periodic GBM Mean"
            )

            ax.fill_between(
                test_month["Date"].dt.day,
                lower,
                upper,
                color="green",
                alpha=0.15,
                label="5%-95% Band"
            )

            ax.set_title(month_names[month - 1], fontsize=11, pad=8)
            ax.set_xlabel("Day")
            ax.set_ylabel("Price")
            ax.set_ylim(y_min, y_max)
            ax.grid(True, linestyle="--", alpha=0.4)

            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1.1)

        fig.suptitle(
            f"{ticker}: Periodic-Correction GBM Month-by-Month Forecast "
            f"({year}, 252-Day Window)",
            fontsize=18,
            fontweight="bold",
            y=0.98
        )

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.945),
            ncol=3,
            frameon=True
        )

        plt.subplots_adjust(
            top=0.90,
            hspace=0.45,
            wspace=0.25
        )

        plt.show()

gbm_periodic_252_monthly_results_df = pd.DataFrame(gbm_periodic_252_monthly_results)
gbm_periodic_252_monthly_results_df.head()

**Linear Regression price-based test results**

Trains a Linear Regression model on next‑day returns using engineered features, predicts next‑day returns on the test set, converts those returns into predicted prices, compares them with actual next‑day prices, and records MAE/RMSE for each ticker.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

lr_price_results = []

for ticker in tickers:

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_test = test[feature_cols]
    y_test_return = test[target_col]

    # Train Linear Regression on returns
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    # Predict next-day return
    pred_return = lr_model.predict(X_test)

    # Convert predicted return into predicted next-day price
    pred_next_price = test["Close"].values * np.exp(pred_return)

    # Actual next-day price from actual next-day return
    actual_next_price = test["Close"].values * np.exp(y_test_return.values)

    mae = mean_absolute_error(actual_next_price, pred_next_price)
    rmse = np.sqrt(mean_squared_error(actual_next_price, pred_next_price))

    lr_price_results.append({
        "ticker": ticker,
        "lr_price_mae": mae,
        "lr_price_rmse": rmse
    })

lr_price_results_df = pd.DataFrame(lr_price_results)
lr_price_results_df

Build final GBM comparison table

Combines Linear Regression results with basic GBM and the best periodic‑GBM window so you can compare all three models side‑by‑side.

In [ ]:
model_comparison = lr_price_results_df.merge(
    gbm_monthly_summary[[
        "ticker",
        "avg_monthly_mae",
        "avg_monthly_rmse"
    ]],
    on="ticker",
    how="inner"
).merge(
    best_window_by_ticker[[
        "ticker",
        "window_size",
        "avg_mae",
        "avg_rmse"
    ]],
    on="ticker",
    how="inner"
)

model_comparison = model_comparison.rename(columns={
    "avg_monthly_mae": "basic_gbm_monthly_mae",
    "avg_monthly_rmse": "basic_gbm_monthly_rmse",
    "window_size": "best_periodic_window",
    "avg_mae": "periodic_gbm_mae",
    "avg_rmse": "periodic_gbm_rmse"
})

model_comparison

Find the best model by MAE and RMSE

Compares LR, basic GBM, and periodic GBM for each stock and marks which model has the lowest MAE and RMSE.

In [ ]:
mae_cols = [
    "lr_price_mae",
    "basic_gbm_monthly_mae",
    "periodic_gbm_mae"
]

rmse_cols = [
    "lr_price_rmse",
    "basic_gbm_monthly_rmse",
    "periodic_gbm_rmse"
]

model_comparison["best_model_mae"] = model_comparison[mae_cols].idxmin(axis=1)
model_comparison["best_model_rmse"] = model_comparison[rmse_cols].idxmin(axis=1)

model_comparison

Cleaner final table

Shows each stock’s MAE and RMSE for Linear Regression, basic GBM, and the best periodic‑GBM window, plus which model performed best.

In [ ]:
final_model_comparison = model_comparison[[
    "ticker",
    "lr_price_mae",
    "basic_gbm_monthly_mae",
    "periodic_gbm_mae",
    "best_model_mae",
    "lr_price_rmse",
    "basic_gbm_monthly_rmse",
    "periodic_gbm_rmse",
    "best_model_rmse",
    "best_periodic_window"
]]

final_model_comparison

Saves the final comparison of all three models into a CSV file.

In [ ]:
final_model_comparison.to_csv(
    "lr_vs_basic_gbm_vs_periodic_gbm_comparison.csv",
    index=False
)

final_model_comparison

MAE comparison bar chart

Builds a grouped bar chart where each ticker has three bars:
• Linear Regression MAE
• Basic monthly GBM MAE
• Best periodic‑window GBM MAE
This layout makes it easy to visually compare model accuracy across all stocks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(final_model_comparison["ticker"]))
width = 0.25

plt.figure(figsize=(14, 6))

plt.bar(
    x - width,
    final_model_comparison["lr_price_mae"],
    width,
    label="Linear Regression"
)

plt.bar(
    x,
    final_model_comparison["basic_gbm_monthly_mae"],
    width,
    label="Basic Monthly GBM"
)

plt.bar(
    x + width,
    final_model_comparison["periodic_gbm_mae"],
    width,
    label="Periodic GBM"
)

plt.xticks(x, final_model_comparison["ticker"])
plt.xlabel("Ticker")
plt.ylabel("MAE")
plt.title("Model Comparison by MAE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

RMSE comparison bar chart

Creates a three‑bar comparison for each ticker:
• Linear Regression RMSE
• Basic monthly GBM RMSE
• Best periodic‑window GBM RMSE
This grouped layout makes it easy to visually compare model performance and identify which model produces the lowest error.

In [ ]:
x = np.arange(len(final_model_comparison["ticker"]))
width = 0.25

plt.figure(figsize=(14, 6))

plt.bar(
    x - width,
    final_model_comparison["lr_price_rmse"],
    width,
    label="Linear Regression"
)

plt.bar(
    x,
    final_model_comparison["basic_gbm_monthly_rmse"],
    width,
    label="Basic Monthly GBM"
)

plt.bar(
    x + width,
    final_model_comparison["periodic_gbm_rmse"],
    width,
    label="Periodic GBM"
)

plt.xticks(x, final_model_comparison["ticker"])
plt.xlabel("Ticker")
plt.ylabel("RMSE")
plt.title("Model Comparison by RMSE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

##Random Walk Hypothesis (RWH)

This is just the container for the Random Walk results.

In [ ]:
random_walk_results = []

Random Walk price baseline

Predicts tomorrow’s price as today’s price, compares it to the real next‑day price, and records MAE/RMSE for each stock.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

random_walk_results = []

for ticker in tickers:

    test = splits[ticker]["test"].copy()

    # Random Walk prediction:
    # Tomorrow's price is predicted to be today's close price
    pred_next_price = test["Close"].values

    # Actual next-day price using actual next-day return
    actual_next_price = test["Close"].values * np.exp(test[target_col].values)

    mae = mean_absolute_error(actual_next_price, pred_next_price)
    rmse = np.sqrt(mean_squared_error(actual_next_price, pred_next_price))

    random_walk_results.append({
        "ticker": ticker,
        "random_walk_mae": mae,
        "random_walk_rmse": rmse
    })

random_walk_results_df = pd.DataFrame(random_walk_results)
random_walk_results_df

Add Random Walk to your model comparison table

In [ ]:
model_comparison_with_rw = random_walk_results_df.merge(
    final_model_comparison,
    on="ticker",
    how="inner"
)

model_comparison_with_rw

Cleaner comparison table

In [ ]:
comparison_with_rw_clean = model_comparison_with_rw[[
    "ticker",
    "random_walk_mae",
    "lr_price_mae",
    "basic_gbm_monthly_mae",
    "periodic_gbm_mae",
    "random_walk_rmse",
    "lr_price_rmse",
    "basic_gbm_monthly_rmse",
    "periodic_gbm_rmse",
    "best_periodic_window"
]]

comparison_with_rw_clean

Find best model including Random Walk

In [ ]:
mae_cols = [
    "random_walk_mae",
    "lr_price_mae",
    "basic_gbm_monthly_mae",
    "periodic_gbm_mae"
]

rmse_cols = [
    "random_walk_rmse",
    "lr_price_rmse",
    "basic_gbm_monthly_rmse",
    "periodic_gbm_rmse"
]

comparison_with_rw_clean = comparison_with_rw_clean.copy()

comparison_with_rw_clean["best_model_mae"] = (
    comparison_with_rw_clean[mae_cols].idxmin(axis=1)
)

comparison_with_rw_clean["best_model_rmse"] = (
    comparison_with_rw_clean[rmse_cols].idxmin(axis=1)
)

comparison_with_rw_clean

Save the comparison table

In [ ]:
comparison_with_rw_clean.to_csv(
    "random_walk_lr_gbm_comparison.csv",
    index=False
)

comparison_with_rw_clean

MAE chart including Random Walk

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(comparison_with_rw_clean["ticker"]))
width = 0.20

plt.figure(figsize=(15, 6))

plt.bar(
    x - 1.5 * width,
    comparison_with_rw_clean["random_walk_mae"],
    width,
    label="Random Walk"
)

plt.bar(
    x - 0.5 * width,
    comparison_with_rw_clean["lr_price_mae"],
    width,
    label="Linear Regression"
)

plt.bar(
    x + 0.5 * width,
    comparison_with_rw_clean["basic_gbm_monthly_mae"],
    width,
    label="Basic Monthly GBM"
)

plt.bar(
    x + 1.5 * width,
    comparison_with_rw_clean["periodic_gbm_mae"],
    width,
    label="Periodic GBM"
)

plt.xticks(x, comparison_with_rw_clean["ticker"])
plt.xlabel("Ticker")
plt.ylabel("MAE")
plt.title("Model Comparison by MAE: Random Walk vs LR vs GBM")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

RMSE chart including Random Walk

In [ ]:
x = np.arange(len(comparison_with_rw_clean["ticker"]))
width = 0.20

plt.figure(figsize=(15, 6))

plt.bar(
    x - 1.5 * width,
    comparison_with_rw_clean["random_walk_rmse"],
    width,
    label="Random Walk"
)

plt.bar(
    x - 0.5 * width,
    comparison_with_rw_clean["lr_price_rmse"],
    width,
    label="Linear Regression"
)

plt.bar(
    x + 0.5 * width,
    comparison_with_rw_clean["basic_gbm_monthly_rmse"],
    width,
    label="Basic Monthly GBM"
)

plt.bar(
    x + 1.5 * width,
    comparison_with_rw_clean["periodic_gbm_rmse"],
    width,
    label="Periodic GBM"
)

plt.xticks(x, comparison_with_rw_clean["ticker"])
plt.xlabel("Ticker")
plt.ylabel("RMSE")
plt.title("Model Comparison by RMSE: Random Walk vs LR vs GBM")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

##Evaluation Metrics Update

This block is a quick metric toolkit. It calculates the main errors (MAE, RMSE), the percentage ones (MAPE, sMAPE), a simple bias check, MASE against a naïve baseline, and directional accuracy. Everything runs through evaluate_forecast() so every model gets scored the same way.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0

    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100


def mean_absolute_scaled_error(y_true, y_pred, naive_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    naive_pred = np.array(naive_pred)

    model_mae = np.mean(np.abs(y_true - y_pred))
    naive_mae = np.mean(np.abs(y_true - naive_pred))

    if naive_mae == 0:
        return np.nan

    return model_mae / naive_mae


def directional_accuracy_from_prices(y_true, y_pred, current_price):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    current_price = np.array(current_price)

    actual_direction = np.sign(y_true - current_price)
    predicted_direction = np.sign(y_pred - current_price)

    return np.mean(actual_direction == predicted_direction)


def evaluate_forecast(y_true, y_pred, current_price=None, naive_pred=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    results = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "sMAPE": symmetric_mean_absolute_percentage_error(y_true, y_pred),
        "Mean_Error": np.mean(y_pred - y_true)
    }

    if naive_pred is not None:
        results["MASE"] = mean_absolute_scaled_error(y_true, y_pred, naive_pred)
    else:
        results["MASE"] = np.nan

    if current_price is not None:
        results["Directional_Accuracy"] = directional_accuracy_from_prices(
            y_true,
            y_pred,
            current_price
        )
    else:
        results["Directional_Accuracy"] = np.nan

    return results

Re-evaluate Random Walk and Linear Regression using full metrics

In [ ]:
from sklearn.linear_model import LinearRegression

full_baseline_metrics = []

for ticker in tickers:

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_test = test[feature_cols]

    current_price = test["Close"].values

    # Actual next-day price
    actual_next_price = test["Close"].values * np.exp(test[target_col].values)

    # Random Walk prediction: tomorrow = today
    random_walk_pred = current_price

    # Linear Regression prediction
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    lr_pred_return = lr_model.predict(X_test)
    lr_pred_price = current_price * np.exp(lr_pred_return)

    # Evaluate Random Walk
    rw_metrics = evaluate_forecast(
        y_true=actual_next_price,
        y_pred=random_walk_pred,
        current_price=current_price,
        naive_pred=random_walk_pred
    )

    full_baseline_metrics.append({
        "ticker": ticker,
        "model": "Random Walk",
        **rw_metrics
    })

    # Evaluate Linear Regression
    lr_metrics = evaluate_forecast(
        y_true=actual_next_price,
        y_pred=lr_pred_price,
        current_price=current_price,
        naive_pred=random_walk_pred
    )

    full_baseline_metrics.append({
        "ticker": ticker,
        "model": "Linear Regression",
        **lr_metrics
    })

full_baseline_metrics_df = pd.DataFrame(full_baseline_metrics)
full_baseline_metrics_df

Check average performance by model

In [ ]:
baseline_model_average = full_baseline_metrics_df.groupby("model").agg(
    avg_MAE=("MAE", "mean"),
    avg_RMSE=("RMSE", "mean"),
    avg_MAPE=("MAPE", "mean"),
    avg_sMAPE=("sMAPE", "mean"),
    avg_MASE=("MASE", "mean"),
    avg_Mean_Error=("Mean_Error", "mean"),
    avg_Directional_Accuracy=("Directional_Accuracy", "mean")
).reset_index()

baseline_model_average

Store prediction values for residual analysis

Now we store actual and predicted values, so we can plot residuals and QQ plots.

This part just builds a simple baseline dataset for each stock. I generate Random Walk predictions (tomorrow = today) and Linear Regression predictions (predict return → convert to price), then store the dates, actual prices, predictions, and residuals together. It gives me a straightforward baseline I can use later when comparing models.

In [ ]:
baseline_predictions = []

for ticker in tickers:

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]

    X_test = test[feature_cols]

    current_price = test["Close"].values
    actual_next_price = test["Close"].values * np.exp(test[target_col].values)

    # Random Walk
    random_walk_pred = current_price

    # Linear Regression
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    lr_pred_return = lr_model.predict(X_test)
    lr_pred_price = current_price * np.exp(lr_pred_return)

    temp_rw = pd.DataFrame({
        "Date": test["Date"].values,
        "ticker": ticker,
        "model": "Random Walk",
        "current_price": current_price,
        "actual": actual_next_price,
        "prediction": random_walk_pred
    })

    temp_lr = pd.DataFrame({
        "Date": test["Date"].values,
        "ticker": ticker,
        "model": "Linear Regression",
        "current_price": current_price,
        "actual": actual_next_price,
        "prediction": lr_pred_price
    })

    baseline_predictions.append(temp_rw)
    baseline_predictions.append(temp_lr)

baseline_predictions_df = pd.concat(baseline_predictions, ignore_index=True)
baseline_predictions_df["residual"] = (
    baseline_predictions_df["prediction"] - baseline_predictions_df["actual"]
)

baseline_predictions_df.head()

Residual Summary

In [ ]:
residual_summary = baseline_predictions_df.groupby(["ticker", "model"]).agg(
    mean_residual=("residual", "mean"),
    median_residual=("residual", "median"),
    std_residual=("residual", "std"),
    min_residual=("residual", "min"),
    max_residual=("residual", "max")
).reset_index()

residual_summary

Residual distribution plots

In [ ]:
import matplotlib.pyplot as plt
import math

models_to_plot = ["Random Walk", "Linear Regression"]

for model_name in models_to_plot:

    temp_model = baseline_predictions_df[
        baseline_predictions_df["model"] == model_name
    ].copy()

    cols = 3
    rows = math.ceil(len(tickers) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
    axes = axes.flatten()

    for i, ticker in enumerate(tickers):

        ax = axes[i]

        temp = temp_model[temp_model["ticker"] == ticker]

        ax.hist(temp["residual"], bins=80)
        ax.axvline(0, linewidth=2)
        ax.set_title(f"{ticker} Residuals — {model_name}")
        ax.set_xlabel("Prediction Error")
        ax.set_ylabel("Frequency")
        ax.grid(True, linestyle="--", alpha=0.4)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"Residual Distribution: {model_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

QQ plots for residuals

In [ ]:
import scipy.stats as stats

for model_name in models_to_plot:

    temp_model = baseline_predictions_df[
        baseline_predictions_df["model"] == model_name
    ].copy()

    cols = 3
    rows = math.ceil(len(tickers) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
    axes = axes.flatten()

    for i, ticker in enumerate(tickers):

        ax = axes[i]

        temp = temp_model[temp_model["ticker"] == ticker]

        stats.probplot(temp["residual"], dist="norm", plot=ax)
        ax.set_title(f"{ticker} QQ Plot — {model_name}")
        ax.grid(True, linestyle="--", alpha=0.4)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(f"QQ Plots of Residuals: {model_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

##LSTM without sentiment

LSTM imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import random

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

Sequence creator

In [ ]:
def create_lstm_sequences(df, feature_cols, target_col, sequence_length=30):
    X = []
    y = []
    current_prices = []
    actual_next_prices = []
    dates = []

    df = df.sort_values("Date").reset_index(drop=True)

    feature_values = df[feature_cols].values
    target_values = df[target_col].values
    close_values = df["Close"].values
    date_values = df["Date"].values

    for i in range(sequence_length, len(df)):
        X.append(feature_values[i-sequence_length:i])
        y.append(target_values[i])

        current_price = close_values[i]
        actual_next_price = current_price * np.exp(target_values[i])

        current_prices.append(current_price)
        actual_next_prices.append(actual_next_price)
        dates.append(date_values[i])

    return (
        np.array(X),
        np.array(y),
        np.array(current_prices),
        np.array(actual_next_prices),
        np.array(dates)
    )

Build LSTM model function

In [ ]:
def build_lstm_model(input_shape):
    model = Sequential()

    model.add(
        LSTM(
            units=64,
            return_sequences=True,
            input_shape=input_shape
        )
    )
    model.add(Dropout(0.2))

    model.add(
        LSTM(
            units=32,
            return_sequences=False
        )
    )
    model.add(Dropout(0.2))

    model.add(Dense(16, activation="relu"))
    model.add(Dense(1))

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

Train LSTM for each ticker

In [ ]:
sequence_length = 30
epochs = 30
batch_size = 32

lstm_results = []
lstm_predictions = []
lstm_histories = {}

for ticker in tickers:

    print(f"\nTraining LSTM for {ticker}")

    train = splits[ticker]["train"].copy()
    val = splits[ticker]["val"].copy()
    test = splits[ticker]["test"].copy()

    # Fit scaler on TRAIN only
    scaler = StandardScaler()
    scaler.fit(train[feature_cols])

    train_scaled = train.copy()
    val_scaled = val.copy()
    test_scaled = test.copy()

    train_scaled[feature_cols] = scaler.transform(train[feature_cols])
    val_scaled[feature_cols] = scaler.transform(val[feature_cols])
    test_scaled[feature_cols] = scaler.transform(test[feature_cols])

    # Create sequences separately to avoid boundary leakage
    X_train, y_train, train_current, train_actual, train_dates = create_lstm_sequences(
        train_scaled,
        feature_cols,
        target_col,
        sequence_length
    )

    X_val, y_val, val_current, val_actual, val_dates = create_lstm_sequences(
        val_scaled,
        feature_cols,
        target_col,
        sequence_length
    )

    X_test, y_test, test_current, test_actual, test_dates = create_lstm_sequences(
        test_scaled,
        feature_cols,
        target_col,
        sequence_length
    )

    input_shape = (X_train.shape[1], X_train.shape[2])

    model = build_lstm_model(input_shape)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    lstm_histories[ticker] = history.history

    # Predict next-day return
    pred_return = model.predict(X_test).flatten()

    # Convert predicted return to predicted next-day price
    pred_next_price = test_current * np.exp(pred_return)

    # Random Walk prediction on the same aligned test rows
    random_walk_pred = test_current

    metrics = evaluate_forecast(
        y_true=test_actual,
        y_pred=pred_next_price,
        current_price=test_current,
        naive_pred=random_walk_pred
    )

    lstm_results.append({
        "ticker": ticker,
        "sequence_length": sequence_length,
        **metrics
    })

    temp_pred = pd.DataFrame({
        "Date": test_dates,
        "ticker": ticker,
        "current_price": test_current,
        "actual": test_actual,
        "prediction": pred_next_price,
        "predicted_return": pred_return,
        "actual_return": y_test
    })

    temp_pred["residual"] = temp_pred["prediction"] - temp_pred["actual"]

    lstm_predictions.append(temp_pred)

lstm_results_df = pd.DataFrame(lstm_results)
lstm_predictions_df = pd.concat(lstm_predictions, ignore_index=True)

lstm_results_df

LSTM average result

In [ ]:
lstm_average_results = lstm_results_df.agg({
    "MAE": "mean",
    "RMSE": "mean",
    "MAPE": "mean",
    "sMAPE": "mean",
    "Mean_Error": "mean",
    "MASE": "mean",
    "Directional_Accuracy": "mean"
}).to_frame().T

lstm_average_results

Plot training and validation loss

In [ ]:
cols = 3
rows = math.ceil(len(tickers) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):

    ax = axes[i]

    history = lstm_histories[ticker]

    ax.plot(history["loss"], label="Train Loss")
    ax.plot(history["val_loss"], label="Validation Loss")

    ax.set_title(f"{ticker} LSTM Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("LSTM Training and Validation Loss", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Plot LSTM actual vs predicted

In [ ]:
cols = 3
rows = math.ceil(len(tickers) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):

    ax = axes[i]

    temp = lstm_predictions_df[lstm_predictions_df["ticker"] == ticker].copy()

    ax.plot(
        temp["Date"],
        temp["actual"],
        label="Actual",
        linewidth=1.5
    )

    ax.plot(
        temp["Date"],
        temp["prediction"],
        label="LSTM Prediction",
        linewidth=1.2
    )

    ax.set_title(f"{ticker} LSTM: Actual vs Predicted")
    ax.set_xlabel("Date")
    ax.set_ylabel("Next-Day Price")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("LSTM Actual vs Predicted Next-Day Prices", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

Compare LSTM with Random Walk and Linear Regression

In [ ]:
lstm_baseline_comparison = []

for ticker in tickers:

    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    scaler = StandardScaler()
    scaler.fit(train[feature_cols])

    test_scaled = test.copy()
    test_scaled[feature_cols] = scaler.transform(test[feature_cols])

    X_test_seq, y_test_seq, test_current, test_actual, test_dates = create_lstm_sequences(
        test_scaled,
        feature_cols,
        target_col,
        sequence_length
    )

    # Random Walk on aligned rows
    random_walk_pred = test_current

    rw_metrics = evaluate_forecast(
        y_true=test_actual,
        y_pred=random_walk_pred,
        current_price=test_current,
        naive_pred=random_walk_pred
    )

    lstm_baseline_comparison.append({
        "ticker": ticker,
        "model": "Random Walk",
        **rw_metrics
    })

    # Linear Regression on aligned rows
    lr_model = LinearRegression()
    lr_model.fit(train[feature_cols], train[target_col])

    # Need original unscaled feature rows aligned with sequence dates
    test_aligned = test[test["Date"].isin(test_dates)].copy()

    lr_pred_return = lr_model.predict(test_aligned[feature_cols])
    lr_pred_price = test_current * np.exp(lr_pred_return)

    lr_metrics = evaluate_forecast(
        y_true=test_actual,
        y_pred=lr_pred_price,
        current_price=test_current,
        naive_pred=random_walk_pred
    )

    lstm_baseline_comparison.append({
        "ticker": ticker,
        "model": "Linear Regression",
        **lr_metrics
    })

    # LSTM results from saved predictions
    lstm_temp = lstm_predictions_df[lstm_predictions_df["ticker"] == ticker].copy()

    lstm_metrics = evaluate_forecast(
        y_true=lstm_temp["actual"].values,
        y_pred=lstm_temp["prediction"].values,
        current_price=lstm_temp["current_price"].values,
        naive_pred=lstm_temp["current_price"].values
    )

    lstm_baseline_comparison.append({
        "ticker": ticker,
        "model": "LSTM",
        **lstm_metrics
    })

lstm_baseline_comparison_df = pd.DataFrame(lstm_baseline_comparison)
lstm_baseline_comparison_df

Average comparison: Random Walk vs LR vs LSTM

In [ ]:
lstm_average_comparison = lstm_baseline_comparison_df.groupby("model").agg(
    avg_MAE=("MAE", "mean"),
    avg_RMSE=("RMSE", "mean"),
    avg_MAPE=("MAPE", "mean"),
    avg_sMAPE=("sMAPE", "mean"),
    avg_MASE=("MASE", "mean"),
    avg_Mean_Error=("Mean_Error", "mean"),
    avg_Directional_Accuracy=("Directional_Accuracy", "mean")
).reset_index()

lstm_average_comparison

Best model per ticker

In [ ]:
lstm_best_by_ticker = (
    lstm_baseline_comparison_df
    .sort_values(["ticker", "MAE"])
    .groupby("ticker")
    .first()
    .reset_index()
)

lstm_best_by_ticker

Bar chart comparison by MAE

In [ ]:
comparison_pivot_mae = lstm_baseline_comparison_df.pivot(
    index="ticker",
    columns="model",
    values="MAE"
).reset_index()

x = np.arange(len(comparison_pivot_mae["ticker"]))
width = 0.25

plt.figure(figsize=(14, 6))

plt.bar(
    x - width,
    comparison_pivot_mae["Random Walk"],
    width,
    label="Random Walk"
)

plt.bar(
    x,
    comparison_pivot_mae["Linear Regression"],
    width,
    label="Linear Regression"
)

plt.bar(
    x + width,
    comparison_pivot_mae["LSTM"],
    width,
    label="LSTM"
)

plt.xticks(x, comparison_pivot_mae["ticker"])
plt.xlabel("Ticker")
plt.ylabel("MAE")
plt.title("Random Walk vs Linear Regression vs LSTM by MAE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

Bar chart comparison by RMSE

In [ ]:
comparison_pivot_rmse = lstm_baseline_comparison_df.pivot(
    index="ticker",
    columns="model",
    values="RMSE"
).reset_index()

x = np.arange(len(comparison_pivot_rmse["ticker"]))
width = 0.25

plt.figure(figsize=(14, 6))

plt.bar(
    x - width,
    comparison_pivot_rmse["Random Walk"],
    width,
    label="Random Walk"
)

plt.bar(
    x,
    comparison_pivot_rmse["Linear Regression"],
    width,
    label="Linear Regression"
)

plt.bar(
    x + width,
    comparison_pivot_rmse["LSTM"],
    width,
    label="LSTM"
)

plt.xticks(x, comparison_pivot_rmse["ticker"])
plt.xlabel("Ticker")
plt.ylabel("RMSE")
plt.title("Random Walk vs Linear Regression vs LSTM by RMSE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

Save LSTM outputs

In [ ]:
lstm_results_df.to_csv("lstm_results_no_sentiment.csv", index=False)
lstm_predictions_df.to_csv("lstm_predictions_no_sentiment.csv", index=False)
lstm_baseline_comparison_df.to_csv("random_walk_lr_lstm_comparison.csv", index=False)
lstm_average_comparison.to_csv("random_walk_lr_lstm_average_comparison.csv", index=False)

print("Saved LSTM outputs.")

## Implementing sentiment in the LSTM models

In [ ]:
gdelt = pd.read_csv("final_gdelt_sentiment_2015_2025.csv")

Load GDELT sentiment file

In [ ]:
import os

print(os.path.exists("/content/final_gdelt_sentiment_2015_2025.csv"))

In [ ]:
import pandas as pd
import numpy as np
import os

gdelt_path = "/content/final_gdelt_sentiment_2015_2025.csv"

gdelt_sentiment = pd.read_csv(gdelt_path)

gdelt_sentiment["date"] = pd.to_datetime(gdelt_sentiment["date"])
gdelt_sentiment["ticker"] = gdelt_sentiment["ticker"].astype(str).str.upper()

gdelt_sentiment.head()

Check GDELT structure

In [ ]:
print(gdelt_sentiment.shape)
print(gdelt_sentiment.columns.tolist())

gdelt_sentiment.groupby("ticker").agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    rows=("date", "count"),
    total_news=("news_count", "sum")
).reset_index()

Prepare stock feature data for merging

In [ ]:
print(gdelt_sentiment.shape)
print(gdelt_sentiment.columns.tolist())

gdelt_sentiment.groupby("ticker").agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    rows=("date", "count"),
    total_news=("news_count", "sum")
).reset_index()

Prepare stock feature data for merging

In [ ]:
stock_feature_frames = []

for ticker in tickers:
    temp = featured_data[ticker].copy()
    temp["ticker"] = ticker
    stock_feature_frames.append(temp)

stock_features_all = pd.concat(stock_feature_frames, ignore_index=True)

stock_features_all = stock_features_all.rename(columns={"Date": "date"})
stock_features_all["date"] = pd.to_datetime(stock_features_all["date"])
stock_features_all["ticker"] = stock_features_all["ticker"].astype(str).str.upper()

stock_features_all.head()

Limit stock data to sentiment period

In [ ]:
stock_features_sentiment_period = stock_features_all[
    stock_features_all["date"] >= "2015-01-01"
].copy()

stock_features_sentiment_period.groupby("ticker").agg(
    start_date=("date", "min"),
    end_date=("date", "max"),
    rows=("date", "count")
).reset_index()

Merge stock features with GDELT sentiment

In [ ]:
sentiment_data = stock_features_sentiment_period.merge(
    gdelt_sentiment,
    on=["ticker", "date"],
    how="left"
)

sentiment_data.head()

Fill missing sentiment days

In [ ]:
sentiment_cols_raw = [
    "news_count",
    "sent_mean",
    "sent_std",
    "sent_min",
    "sent_max"
]

for col in sentiment_cols_raw:
    sentiment_data[col] = sentiment_data[col].fillna(0)

sentiment_data[sentiment_cols_raw].isna().sum()

create lagged and rolling sentiment features

In [ ]:
sentiment_data = sentiment_data.sort_values(["ticker", "date"]).reset_index(drop=True)

for ticker in tickers:
    mask = sentiment_data["ticker"] == ticker

    # Lagged sentiment features
    sentiment_data.loc[mask, "sent_mean_lag1"] = sentiment_data.loc[mask, "sent_mean"].shift(1)
    sentiment_data.loc[mask, "sent_mean_lag2"] = sentiment_data.loc[mask, "sent_mean"].shift(2)
    sentiment_data.loc[mask, "sent_mean_lag5"] = sentiment_data.loc[mask, "sent_mean"].shift(5)

    # Lagged news count features
    sentiment_data.loc[mask, "news_count_lag1"] = sentiment_data.loc[mask, "news_count"].shift(1)
    sentiment_data.loc[mask, "news_count_lag2"] = sentiment_data.loc[mask, "news_count"].shift(2)
    sentiment_data.loc[mask, "news_count_lag5"] = sentiment_data.loc[mask, "news_count"].shift(5)

    # Rolling sentiment features using past data only
    sentiment_data.loc[mask, "rolling_sent_7"] = (
        sentiment_data.loc[mask, "sent_mean"]
        .shift(1)
        .rolling(7)
        .mean()
    )

    sentiment_data.loc[mask, "rolling_sent_30"] = (
        sentiment_data.loc[mask, "sent_mean"]
        .shift(1)
        .rolling(30)
        .mean()
    )

    # Rolling news count features using past data only
    sentiment_data.loc[mask, "rolling_news_7"] = (
        sentiment_data.loc[mask, "news_count"]
        .shift(1)
        .rolling(7)
        .mean()
    )

    sentiment_data.loc[mask, "rolling_news_30"] = (
        sentiment_data.loc[mask, "news_count"]
        .shift(1)
        .rolling(30)
        .mean()
    )

sentiment_data.head()

In [ ]:
sentiment_data = sentiment_data.replace([np.inf, -np.inf], np.nan)
sentiment_data = sentiment_data.dropna().reset_index(drop=True)

sentiment_data.isna().sum().sum()

define sentiment feature columns

In [ ]:
sentiment_feature_cols = feature_cols + [
    "news_count",
    "sent_mean",
    "sent_std",
    "sent_min",
    "sent_max",
    "sent_mean_lag1",
    "sent_mean_lag2",
    "sent_mean_lag5",
    "news_count_lag1",
    "news_count_lag2",
    "news_count_lag5",
    "rolling_sent_7",
    "rolling_sent_30",
    "rolling_news_7",
    "rolling_news_30"
]

len(sentiment_feature_cols), sentiment_feature_cols

Rebuild sentiment splits

In [ ]:
sentiment_splits = {}

for ticker in tickers:
    df = sentiment_data[sentiment_data["ticker"] == ticker].copy()
    df = df.sort_values("date").reset_index(drop=True)

    train = df[df["date"] < "2019-01-01"]
    val = df[(df["date"] >= "2019-01-01") & (df["date"] < "2022-01-01")]
    test = df[df["date"] >= "2022-01-01"]

    sentiment_splits[ticker] = {
        "train": train.reset_index(drop=True),
        "val": val.reset_index(drop=True),
        "test": test.reset_index(drop=True)
    }

Leakage check

In [ ]:
for ticker in sentiment_splits:
    train = sentiment_splits[ticker]["train"]
    val = sentiment_splits[ticker]["val"]
    test = sentiment_splits[ticker]["test"]

    print(f"\n{ticker}")
    print("Train:", train["date"].min(), "to", train["date"].max(), train.shape)
    print("Val:  ", val["date"].min(), "to", val["date"].max(), val.shape)
    print("Test: ", test["date"].min(), "to", test["date"].max(), test.shape)

    assert train["date"].max() < val["date"].min(), f"{ticker}: Train overlaps validation"
    assert val["date"].max() < test["date"].min(), f"{ticker}: Validation overlaps test"

    assert train["date"].is_unique, f"{ticker}: Duplicate train dates"
    assert val["date"].is_unique, f"{ticker}: Duplicate validation dates"
    assert test["date"].is_unique, f"{ticker}: Duplicate test dates"

print("\nAll sentiment leakage checks passed.")

Sentiment dataset summary

In [ ]:
sentiment_summary = []

for ticker in tickers:
    train = sentiment_splits[ticker]["train"]
    val = sentiment_splits[ticker]["val"]
    test = sentiment_splits[ticker]["test"]

    all_df = pd.concat([train, val, test], ignore_index=True)

    sentiment_summary.append({
        "ticker": ticker,
        "start_date": all_df["date"].min(),
        "end_date": all_df["date"].max(),
        "rows": len(all_df),
        "train_rows": len(train),
        "val_rows": len(val),
        "test_rows": len(test),
        "total_news": all_df["news_count"].sum(),
        "avg_sent_mean": all_df["sent_mean"].mean(),
        "avg_news_count": all_df["news_count"].mean(),
        "missing_values": all_df[sentiment_feature_cols + [target_col]].isna().sum().sum()
    })

sentiment_summary_df = pd.DataFrame(sentiment_summary)
sentiment_summary_df

## Creating Models with sentiment

Create sentiment LSTM sequence function

In [ ]:
def create_sentiment_lstm_sequences(df, feature_cols, target_col, sequence_length=30):
    X = []
    y = []
    current_prices = []
    actual_next_prices = []
    dates = []

    df = df.sort_values("date").reset_index(drop=True)

    feature_values = df[feature_cols].values
    target_values = df[target_col].values
    close_values = df["Close"].values
    date_values = df["date"].values

    for i in range(sequence_length, len(df)):
        X.append(feature_values[i-sequence_length:i])
        y.append(target_values[i])

        current_price = close_values[i]
        actual_next_price = current_price * np.exp(target_values[i])

        current_prices.append(current_price)
        actual_next_prices.append(actual_next_price)
        dates.append(date_values[i])

    return (
        np.array(X),
        np.array(y),
        np.array(current_prices),
        np.array(actual_next_prices),
        np.array(dates)
    )

Build cleaner LSTM model function

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

def build_sentiment_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),

        LSTM(
            units=64,
            return_sequences=True
        ),
        Dropout(0.2),

        LSTM(
            units=32,
            return_sequences=False
        ),
        Dropout(0.2),

        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    return model

Train LSTM + Sentiment for each ticker

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0

    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100


def mean_absolute_scaled_error(y_true, y_pred, naive_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    naive_pred = np.array(naive_pred)

    model_mae = np.mean(np.abs(y_true - y_pred))
    naive_mae = np.mean(np.abs(y_true - naive_pred))

    if naive_mae == 0:
        return np.nan

    return model_mae / naive_mae


def directional_accuracy_from_prices(y_true, y_pred, current_price):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    current_price = np.array(current_price)

    actual_direction = np.sign(y_true - current_price)
    predicted_direction = np.sign(y_pred - current_price)

    return np.mean(actual_direction == predicted_direction)


def evaluate_forecast(y_true, y_pred, current_price=None, naive_pred=None):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    results = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "sMAPE": symmetric_mean_absolute_percentage_error(y_true, y_pred),
        "Mean_Error": np.mean(y_pred - y_true)
    }

    if naive_pred is not None:
        results["MASE"] = mean_absolute_scaled_error(y_true, y_pred, naive_pred)
    else:
        results["MASE"] = np.nan

    if current_price is not None:
        results["Directional_Accuracy"] = directional_accuracy_from_prices(
            y_true,
            y_pred,
            current_price
        )
    else:
        results["Directional_Accuracy"] = np.nan

    return results

In [ ]:
print("evaluate_forecast exists:", "evaluate_forecast" in globals())

In [ ]:
import random
import tensorflow as tf

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

sequence_length = 30
epochs = 30
batch_size = 32

lstm_sentiment_results = []
lstm_sentiment_predictions = []
lstm_sentiment_histories = {}

for ticker in tickers:

    print(f"\nTraining LSTM + Sentiment for {ticker}")

    train = sentiment_splits[ticker]["train"].copy()
    val = sentiment_splits[ticker]["val"].copy()
    test = sentiment_splits[ticker]["test"].copy()

    # Fit scaler on TRAIN only
    scaler = StandardScaler()
    scaler.fit(train[sentiment_feature_cols])

    train_scaled = train.copy()
    val_scaled = val.copy()
    test_scaled = test.copy()

    train_scaled[sentiment_feature_cols] = scaler.transform(train[sentiment_feature_cols])
    val_scaled[sentiment_feature_cols] = scaler.transform(val[sentiment_feature_cols])
    test_scaled[sentiment_feature_cols] = scaler.transform(test[sentiment_feature_cols])

    # Create sequences separately to avoid boundary leakage
    X_train, y_train, train_current, train_actual, train_dates = create_sentiment_lstm_sequences(
        train_scaled,
        sentiment_feature_cols,
        target_col,
        sequence_length
    )

    X_val, y_val, val_current, val_actual, val_dates = create_sentiment_lstm_sequences(
        val_scaled,
        sentiment_feature_cols,
        target_col,
        sequence_length
    )

    X_test, y_test, test_current, test_actual, test_dates = create_sentiment_lstm_sequences(
        test_scaled,
        sentiment_feature_cols,
        target_col,
        sequence_length
    )

    input_shape = (X_train.shape[1], X_train.shape[2])

    model = build_sentiment_lstm_model(input_shape)

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    lstm_sentiment_histories[ticker] = history.history

    # Predict next-day return
    pred_return = model.predict(X_test).flatten()

    # Convert predicted return into predicted next-day price
    pred_next_price = test_current * np.exp(pred_return)

    # Random Walk prediction on same aligned rows
    random_walk_pred = test_current

    metrics = evaluate_forecast(
        y_true=test_actual,
        y_pred=pred_next_price,
        current_price=test_current,
        naive_pred=random_walk_pred
    )

    lstm_sentiment_results.append({
        "ticker": ticker,
        "sequence_length": sequence_length,
        **metrics
    })

    temp_pred = pd.DataFrame({
        "Date": test_dates,
        "ticker": ticker,
        "current_price": test_current,
        "actual": test_actual,
        "prediction": pred_next_price,
        "predicted_return": pred_return,
        "actual_return": y_test
    })

    temp_pred["residual"] = temp_pred["prediction"] - temp_pred["actual"]

    lstm_sentiment_predictions.append(temp_pred)

lstm_sentiment_results_df = pd.DataFrame(lstm_sentiment_results)
lstm_sentiment_predictions_df = pd.concat(lstm_sentiment_predictions, ignore_index=True)

lstm_sentiment_results_df

Checking to confirm if required variables exist and data leakage insurance

Check required variables exist

In [ ]:
required_vars = [
    "tickers",
    "splits",
    "feature_cols",
    "target_col",
    "evaluate_forecast",
    "sentiment_splits",
    "sentiment_feature_cols",
    "lstm_sentiment_results_df",
    "lstm_sentiment_predictions_df",
    "lstm_sentiment_histories"
]

for var in required_vars:
    print(var, "exists:", var in globals())

Check sentiment split leakage

In [ ]:
for ticker in sentiment_splits:
    train = sentiment_splits[ticker]["train"]
    val = sentiment_splits[ticker]["val"]
    test = sentiment_splits[ticker]["test"]

    print(f"\n{ticker}")
    print("Train:", train["date"].min(), "to", train["date"].max())
    print("Val:  ", val["date"].min(), "to", val["date"].max())
    print("Test: ", test["date"].min(), "to", test["date"].max())

    assert train["date"].max() < val["date"].min(), f"{ticker}: train overlaps validation"
    assert val["date"].max() < test["date"].min(), f"{ticker}: validation overlaps test"

print("\nSentiment split leakage check passed.")

Check missing values in sentiment model inputs

In [ ]:
for ticker in sentiment_splits:
    for split_name in ["train", "val", "test"]:
        df = sentiment_splits[ticker][split_name]

        missing_count = df[sentiment_feature_cols + [target_col]].isna().sum().sum()
        inf_count = np.isinf(df[sentiment_feature_cols + [target_col]]).sum().sum()

        print(f"{ticker} {split_name} | Missing: {missing_count} | Inf: {inf_count}")

Check LSTM + Sentiment prediction outputs

In [ ]:
print("lstm_sentiment_results_df shape:", lstm_sentiment_results_df.shape)
print("lstm_sentiment_predictions_df shape:", lstm_sentiment_predictions_df.shape)

print("\nTickers in results:")
print(lstm_sentiment_results_df["ticker"].unique())

print("\nTickers in predictions:")
print(lstm_sentiment_predictions_df["ticker"].unique())

print("\nMissing values in predictions:")
print(lstm_sentiment_predictions_df.isna().sum())

In [ ]:
import os

save_path = "/content/drive/MyDrive/FYP_outputs"
os.makedirs(save_path, exist_ok=True)

lstm_sentiment_results_df.to_csv(
    f"{save_path}/lstm_sentiment_results.csv",
    index=False
)

lstm_sentiment_predictions_df.to_csv(
    f"{save_path}/lstm_sentiment_predictions.csv",
    index=False
)

print("Saved LSTM + Sentiment results and predictions to Google Drive.")

In [ ]:
print("lstm_results_df exists:", "lstm_results_df" in globals())
print("lstm_predictions_df exists:", "lstm_predictions_df" in globals())

Check no-sentiment LSTM predictions are clean

In [ ]:
print("lstm_results_df shape:", lstm_results_df.shape)
print("lstm_predictions_df shape:", lstm_predictions_df.shape)

print("\nTickers in LSTM results:")
print(lstm_results_df["ticker"].unique())

print("\nTickers in LSTM predictions:")
print(lstm_predictions_df["ticker"].unique())

print("\nMissing values in LSTM predictions:")
print(lstm_predictions_df.isna().sum())

Check date columns are proper datetime

In [ ]:
lstm_predictions_df["Date"] = pd.to_datetime(lstm_predictions_df["Date"])
lstm_sentiment_predictions_df["Date"] = pd.to_datetime(lstm_sentiment_predictions_df["Date"])

print(lstm_predictions_df["Date"].min(), lstm_predictions_df["Date"].max())
print(lstm_sentiment_predictions_df["Date"].min(), lstm_sentiment_predictions_df["Date"].max())

Broader comparison: Random Walk, LR, LSTM, LSTM + Sentiment

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

lstm_sentiment_full_comparison = []

for ticker in tickers:

    # LSTM + Sentiment prediction rows
    sent_temp = lstm_sentiment_predictions_df[
        lstm_sentiment_predictions_df["ticker"] == ticker
    ].copy()

    sent_temp["Date"] = pd.to_datetime(sent_temp["Date"])
    sent_temp = sent_temp.sort_values("Date").reset_index(drop=True)

    aligned_dates = sent_temp["Date"]

    current_price = sent_temp["current_price"].values
    actual = sent_temp["actual"].values

    # -----------------------------
    # Random Walk
    # -----------------------------
    rw_pred = current_price

    rw_metrics = evaluate_forecast(
        y_true=actual,
        y_pred=rw_pred,
        current_price=current_price,
        naive_pred=rw_pred
    )

    lstm_sentiment_full_comparison.append({
        "ticker": ticker,
        "model": "Random Walk",
        **rw_metrics
    })

    # -----------------------------
    # Linear Regression
    # -----------------------------
    train_no_sent = splits[ticker]["train"].copy()
    test_no_sent = splits[ticker]["test"].copy()

    train_no_sent["Date"] = pd.to_datetime(train_no_sent["Date"])
    test_no_sent["Date"] = pd.to_datetime(test_no_sent["Date"])

    test_no_sent_aligned = (
        test_no_sent
        .set_index("Date")
        .reindex(aligned_dates)
        .reset_index()
        .rename(columns={"index": "Date"})
    )

    assert len(test_no_sent_aligned) == len(sent_temp), f"{ticker}: LR alignment length mismatch"
    assert test_no_sent_aligned[feature_cols].isna().sum().sum() == 0, f"{ticker}: Missing LR aligned features"

    lr_model = LinearRegression()
    lr_model.fit(train_no_sent[feature_cols], train_no_sent[target_col])

    lr_pred_return = lr_model.predict(test_no_sent_aligned[feature_cols])
    lr_pred_price = current_price * np.exp(lr_pred_return)

    lr_metrics = evaluate_forecast(
        y_true=actual,
        y_pred=lr_pred_price,
        current_price=current_price,
        naive_pred=rw_pred
    )

    lstm_sentiment_full_comparison.append({
        "ticker": ticker,
        "model": "Linear Regression",
        **lr_metrics
    })

    # -----------------------------
    # LSTM without sentiment
    # -----------------------------
    lstm_no_sent = lstm_predictions_df[
        lstm_predictions_df["ticker"] == ticker
    ].copy()

    lstm_no_sent["Date"] = pd.to_datetime(lstm_no_sent["Date"])

    lstm_no_sent_aligned = (
        lstm_no_sent
        .set_index("Date")
        .reindex(aligned_dates)
        .reset_index()
        .rename(columns={"index": "Date"})
    )

    assert len(lstm_no_sent_aligned) == len(sent_temp), f"{ticker}: LSTM alignment length mismatch"
    assert lstm_no_sent_aligned["prediction"].isna().sum() == 0, f"{ticker}: Missing LSTM aligned predictions"

    lstm_metrics = evaluate_forecast(
        y_true=actual,
        y_pred=lstm_no_sent_aligned["prediction"].values,
        current_price=current_price,
        naive_pred=rw_pred
    )

    lstm_sentiment_full_comparison.append({
        "ticker": ticker,
        "model": "LSTM",
        **lstm_metrics
    })

    # -----------------------------
    # LSTM + Sentiment
    # -----------------------------
    sent_metrics = evaluate_forecast(
        y_true=actual,
        y_pred=sent_temp["prediction"].values,
        current_price=current_price,
        naive_pred=rw_pred
    )

    lstm_sentiment_full_comparison.append({
        "ticker": ticker,
        "model": "LSTM + Sentiment",
        **sent_metrics
    })


lstm_sentiment_full_comparison_df = pd.DataFrame(lstm_sentiment_full_comparison)
lstm_sentiment_full_comparison_df

Average comparison

In [ ]:
lstm_sentiment_average_comparison = lstm_sentiment_full_comparison_df.groupby("model").agg(
    avg_MAE=("MAE", "mean"),
    avg_RMSE=("RMSE", "mean"),
    avg_MAPE=("MAPE", "mean"),
    avg_sMAPE=("sMAPE", "mean"),
    avg_MASE=("MASE", "mean"),
    avg_Mean_Error=("Mean_Error", "mean"),
    avg_Directional_Accuracy=("Directional_Accuracy", "mean")
).reset_index()

lstm_sentiment_average_comparison

Best model per ticker

In [ ]:
lstm_sentiment_best_by_ticker = (
    lstm_sentiment_full_comparison_df
    .sort_values(["ticker", "MAE"])
    .groupby("ticker")
    .first()
    .reset_index()
)

lstm_sentiment_best_by_ticker

In [ ]:
import os

save_path = "/content/drive/MyDrive/FYP_outputs"
os.makedirs(save_path, exist_ok=True)

lstm_sentiment_full_comparison_df.to_csv(
    f"{save_path}/rw_lr_lstm_lstm_sentiment_full_comparison.csv",
    index=False
)

lstm_sentiment_average_comparison.to_csv(
    f"{save_path}/rw_lr_lstm_lstm_sentiment_average_comparison.csv",
    index=False
)

lstm_sentiment_best_by_ticker.to_csv(
    f"{save_path}/rw_lr_lstm_lstm_sentiment_best_by_ticker.csv",
    index=False
)

print("Saved broader comparison outputs.")

## GBM + Sentiment

Check required variables

In [ ]:
required_vars = [
    "tickers",
    "sentiment_splits",
    "simulate_gbm",
    "evaluate_forecast",
    "sentiment_feature_cols",
    "target_col"
]

for var in required_vars:
    print(var, "exists:", var in globals())

GBM + Sentiment model

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)

gbm_sentiment_results = []
gbm_sentiment_predictions = {}

sentiment_signal_col = "rolling_sent_7"

for ticker in tickers:

    train = sentiment_splits[ticker]["train"].copy()
    test = sentiment_splits[ticker]["test"].copy()

    train = train.sort_values("date").reset_index(drop=True)
    test = test.sort_values("date").reset_index(drop=True)

    # Basic GBM parameters from TRAIN only
    mu = train["log_return"].mean() * 252
    sigma = train["log_return"].std() * np.sqrt(252)

    # Estimate sentiment effect from TRAIN only
    beta_model = LinearRegression()
    beta_model.fit(
        train[[sentiment_signal_col]],
        train[target_col]
    )

    beta = beta_model.coef_[0] * 252

    S0 = test["Close"].iloc[0]
    steps = len(test)

    sentiment_signal = test[sentiment_signal_col].values

    n_sims = 1000
    dt = 1 / 252
    simulations = np.zeros((steps, n_sims))

    for sim in range(n_sims):
        prices = [S0]

        for t in range(1, steps):
            Z = np.random.normal()

            mu_adjusted = mu + beta * sentiment_signal[t]

            St = prices[-1] * np.exp(
                (mu_adjusted - 0.5 * sigma**2) * dt
                + sigma * np.sqrt(dt) * Z
            )

            prices.append(St)

        simulations[:, sim] = prices

    gbm_sent_mean = simulations.mean(axis=1)
    lower = np.percentile(simulations, 5, axis=1)
    upper = np.percentile(simulations, 95, axis=1)

    actual = test["Close"].values
    current_price = test["Close"].values
    random_walk_pred = current_price

    metrics = evaluate_forecast(
        y_true=actual,
        y_pred=gbm_sent_mean,
        current_price=current_price,
        naive_pred=random_walk_pred
    )

    gbm_sentiment_results.append({
        "ticker": ticker,
        "mu": mu,
        "sigma": sigma,
        "beta_sentiment": beta,
        "sentiment_signal": sentiment_signal_col,
        **metrics
    })

    gbm_sentiment_predictions[ticker] = pd.DataFrame({
        "Date": test["date"].values,
        "ticker": ticker,
        "actual": actual,
        "prediction": gbm_sent_mean,
        "lower_5": lower,
        "upper_95": upper,
        "sentiment_signal": sentiment_signal
    })

gbm_sentiment_results_df = pd.DataFrame(gbm_sentiment_results)
gbm_sentiment_results_df

Plot GBM + Sentiment vs Actual

In [ ]:
cols = 3
rows = math.ceil(len(tickers) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 4))
axes = axes.flatten()

for i, ticker in enumerate(tickers):

    ax = axes[i]
    temp = gbm_sentiment_predictions[ticker].copy()
    temp["Date"] = pd.to_datetime(temp["Date"])

    ax.plot(
        temp["Date"],
        temp["actual"],
        color="black",
        linewidth=2,
        label="Actual"
    )

    ax.plot(
        temp["Date"],
        temp["prediction"],
        color="purple",
        linewidth=1.5,
        label="GBM + Sentiment Mean"
    )

    ax.fill_between(
        temp["Date"],
        temp["lower_5"],
        temp["upper_95"],
        color="purple",
        alpha=0.15,
        label="5%-95% Band"
    )

    ax.set_title(f"{ticker}: GBM + Sentiment")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price")
    ax.grid(True, linestyle="--", alpha=0.4)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3)

plt.suptitle("GBM + Sentiment Forecast vs Actual", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

GBM + Sentiment year-by-year VS Actual

In [ ]:
import matplotlib.dates as mdates

gbm_sentiment_yearly_results = []

years = sorted(
    sentiment_splits[tickers[0]]["test"]["date"].dt.year.unique()
)

cols = 2
rows = math.ceil(len(years) / cols)

sentiment_signal_col = "rolling_sent_7"

for ticker in tickers:

    train = sentiment_splits[ticker]["train"].copy()
    test = sentiment_splits[ticker]["test"].copy()

    train = train.sort_values("date").reset_index(drop=True)
    test = test.sort_values("date").reset_index(drop=True)

    # Parameters estimated from TRAIN only
    mu = train["log_return"].mean() * 252
    sigma = train["log_return"].std() * np.sqrt(252)

    beta_model = LinearRegression()
    beta_model.fit(
        train[[sentiment_signal_col]],
        train[target_col]
    )

    beta = beta_model.coef_[0] * 252

    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 4.8))
    axes = axes.flatten()

    for i, year in enumerate(years):

        ax = axes[i]
        test_year = test[test["date"].dt.year == year].copy()

        if test_year.empty:
            ax.set_title(f"{year} — No Data")
            ax.axis("off")
            continue

        S0 = test_year["Close"].iloc[0]
        steps = len(test_year)
        sentiment_signal = test_year[sentiment_signal_col].values

        n_sims = 1000
        dt = 1 / 252
        simulations = np.zeros((steps, n_sims))

        for sim in range(n_sims):
            prices = [S0]

            for t in range(1, steps):
                Z = np.random.normal()

                mu_adjusted = mu + beta * sentiment_signal[t]

                St = prices[-1] * np.exp(
                    (mu_adjusted - 0.5 * sigma**2) * dt
                    + sigma * np.sqrt(dt) * Z
                )

                prices.append(St)

            simulations[:, sim] = prices

        gbm_mean = simulations.mean(axis=1)
        lower = np.percentile(simulations, 5, axis=1)
        upper = np.percentile(simulations, 95, axis=1)

        actual = test_year["Close"].values
        current_price = test_year["Close"].values
        random_walk_pred = current_price

        metrics = evaluate_forecast(
            y_true=actual,
            y_pred=gbm_mean,
            current_price=current_price,
            naive_pred=random_walk_pred
        )

        gbm_sentiment_yearly_results.append({
            "ticker": ticker,
            "year": year,
            "mu": mu,
            "sigma": sigma,
            "beta_sentiment": beta,
            **metrics
        })

        ax.plot(
            test_year["date"],
            actual,
            color="black",
            linewidth=2,
            label="Actual"
        )

        ax.plot(
            test_year["date"],
            gbm_mean,
            color="purple",
            linewidth=1.5,
            label="GBM + Sentiment Mean"
        )

        ax.fill_between(
            test_year["date"],
            lower,
            upper,
            color="purple",
            alpha=0.15,
            label="5%-95% Band"
        )

        ax.set_title(f"{ticker} — {year}", fontsize=11, pad=10)
        ax.set_xlabel("Date")
        ax.set_ylabel("Price")
        ax.grid(True, linestyle="--", alpha=0.4)

        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
        ax.tick_params(axis="x", rotation=45)

        for spine in ax.spines.values():
            spine.set_edgecolor("black")
            spine.set_linewidth(1.1)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        f"{ticker}: GBM + Sentiment Year-by-Year Forecast vs Actual",
        fontsize=18,
        fontweight="bold",
        y=0.98
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.94),
        ncol=3,
        frameon=True
    )

    plt.subplots_adjust(
        top=0.86,
        hspace=0.45,
        wspace=0.25
    )

    plt.show()

gbm_sentiment_yearly_results_df = pd.DataFrame(gbm_sentiment_yearly_results)
gbm_sentiment_yearly_results_df.head()

GBM + Sentiment month-by-month VS Actual

In [ ]:
gbm_sentiment_monthly_results = []

month_names = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

sentiment_signal_col = "rolling_sent_7"

for ticker in tickers:

    train = sentiment_splits[ticker]["train"].copy()
    test = sentiment_splits[ticker]["test"].copy()

    train = train.sort_values("date").reset_index(drop=True)
    test = test.sort_values("date").reset_index(drop=True)

    # Parameters estimated from TRAIN only
    mu = train["log_return"].mean() * 252
    sigma = train["log_return"].std() * np.sqrt(252)

    beta_model = LinearRegression()
    beta_model.fit(
        train[[sentiment_signal_col]],
        train[target_col]
    )

    beta = beta_model.coef_[0] * 252

    for year in years:

        test_year = test[test["date"].dt.year == year].copy()

        if test_year.empty:
            continue

        fig, axes = plt.subplots(3, 4, figsize=(20, 12))
        axes = axes.flatten()

        for month in range(1, 13):

            ax = axes[month - 1]
            test_month = test_year[test_year["date"].dt.month == month].copy()

            if test_month.empty:
                ax.set_title(f"{month_names[month - 1]} — No Data")
                ax.axis("off")
                continue

            S0 = test_month["Close"].iloc[0]
            steps = len(test_month)
            sentiment_signal = test_month[sentiment_signal_col].values

            n_sims = 1000
            dt = 1 / 252
            simulations = np.zeros((steps, n_sims))

            for sim in range(n_sims):
                prices = [S0]

                for t in range(1, steps):
                    Z = np.random.normal()

                    mu_adjusted = mu + beta * sentiment_signal[t]

                    St = prices[-1] * np.exp(
                        (mu_adjusted - 0.5 * sigma**2) * dt
                        + sigma * np.sqrt(dt) * Z
                    )

                    prices.append(St)

                simulations[:, sim] = prices

            gbm_mean = simulations.mean(axis=1)
            lower = np.percentile(simulations, 5, axis=1)
            upper = np.percentile(simulations, 95, axis=1)

            actual = test_month["Close"].values
            current_price = test_month["Close"].values
            random_walk_pred = current_price

            metrics = evaluate_forecast(
                y_true=actual,
                y_pred=gbm_mean,
                current_price=current_price,
                naive_pred=random_walk_pred
            )

            gbm_sentiment_monthly_results.append({
                "ticker": ticker,
                "year": year,
                "month": month,
                "mu": mu,
                "sigma": sigma,
                "beta_sentiment": beta,
                **metrics
            })

            ax.plot(
                test_month["date"].dt.day,
                actual,
                color="black",
                linewidth=2,
                label="Actual"
            )

            ax.plot(
                test_month["date"].dt.day,
                gbm_mean,
                color="purple",
                linewidth=1.5,
                label="GBM + Sentiment Mean"
            )

            ax.fill_between(
                test_month["date"].dt.day,
                lower,
                upper,
                color="purple",
                alpha=0.15,
                label="5%-95% Band"
            )

            ax.set_title(month_names[month - 1], fontsize=11, pad=8)
            ax.set_xlabel("Day")
            ax.set_ylabel("Price")
            ax.grid(True, linestyle="--", alpha=0.4)

            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1.1)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.94),
            ncol=3,
            frameon=True
        )

        fig.suptitle(
            f"{ticker}: GBM + Sentiment Month-by-Month Forecast vs Actual ({year})",
            fontsize=18,
            fontweight="bold",
            y=0.985
        )

        plt.subplots_adjust(
            top=0.88,
            hspace=0.50,
            wspace=0.30
        )

        plt.show()

gbm_sentiment_monthly_results_df = pd.DataFrame(gbm_sentiment_monthly_results)
gbm_sentiment_monthly_results_df.head()

Summary cells after yearly/monthly

In [ ]:
gbm_sentiment_yearly_summary = gbm_sentiment_yearly_results_df.groupby("ticker").agg(
    avg_gbm_sent_yearly_mae=("MAE", "mean"),
    avg_gbm_sent_yearly_rmse=("RMSE", "mean"),
    best_gbm_sent_yearly_mae=("MAE", "min"),
    worst_gbm_sent_yearly_mae=("MAE", "max")
).reset_index()

gbm_sentiment_yearly_summary

In [ ]:
gbm_sentiment_monthly_summary = gbm_sentiment_monthly_results_df.groupby("ticker").agg(
    avg_gbm_sent_monthly_mae=("MAE", "mean"),
    avg_gbm_sent_monthly_rmse=("RMSE", "mean"),
    best_gbm_sent_monthly_mae=("MAE", "min"),
    worst_gbm_sent_monthly_mae=("MAE", "max")
).reset_index()

gbm_sentiment_monthly_summary

In [ ]:
import os

save_path = "/content/drive/MyDrive/FYP_outputs"
os.makedirs(save_path, exist_ok=True)

gbm_sentiment_yearly_results_df.to_csv(
    f"{save_path}/gbm_sentiment_yearly_results.csv",
    index=False
)

gbm_sentiment_monthly_results_df.to_csv(
    f"{save_path}/gbm_sentiment_monthly_results.csv",
    index=False
)

gbm_sentiment_yearly_summary.to_csv(
    f"{save_path}/gbm_sentiment_yearly_summary.csv",
    index=False
)

gbm_sentiment_monthly_summary.to_csv(
    f"{save_path}/gbm_sentiment_monthly_summary.csv",
    index=False
)

print("Saved GBM + Sentiment yearly and monthly outputs.")

##Periodic GBM + Sentiment

Check required variables

In [ ]:
required_vars = [
    "tickers",
    "sentiment_splits",
    "sentiment_data",
    "simulate_gbm",
    "evaluate_forecast",
    "target_col"
]

for var in required_vars:
    print(var, "exists:", var in globals())

Periodic GBM + Sentiment month-by-month

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

np.random.seed(42)

window_size = 252
sentiment_signal_col = "rolling_sent_7"

month_names = [
    "Jan", "Feb", "Mar", "Apr",
    "May", "Jun", "Jul", "Aug",
    "Sep", "Oct", "Nov", "Dec"
]

gbm_periodic_sentiment_results = []
gbm_periodic_sentiment_predictions = []

for ticker in tickers:

    print(f"\nRunning Periodic GBM + Sentiment for {ticker}")

    df_all = sentiment_data[
        sentiment_data["ticker"] == ticker
    ].copy()

    df_all = df_all.sort_values("date").reset_index(drop=True)

    test = sentiment_splits[ticker]["test"].copy()
    test = test.sort_values("date").reset_index(drop=True)

    test_years = sorted(test["date"].dt.year.unique())

    for year in test_years:

        test_year = test[test["date"].dt.year == year].copy()

        if test_year.empty:
            continue

        fig, axes = plt.subplots(3, 4, figsize=(20, 12))
        axes = axes.flatten()

        for month in range(1, 13):

            ax = axes[month - 1]

            test_month = test_year[
                test_year["date"].dt.month == month
            ].copy()

            if test_month.empty:
                ax.set_title(f"{month_names[month - 1]} — No Data")
                ax.axis("off")
                continue

            forecast_start_date = test_month["date"].iloc[0]

            # Past data only
            historical_window = df_all[
                df_all["date"] < forecast_start_date
            ].tail(window_size).copy()

            if len(historical_window) < window_size:
                ax.set_title(f"{month_names[month - 1]} — Not enough history")
                ax.axis("off")
                continue

            # Rolling parameters from past data only
            mu = historical_window["log_return"].mean() * 252
            sigma = historical_window["log_return"].std() * np.sqrt(252)

            # Sentiment effect from past data only
            beta_model = LinearRegression()
            beta_model.fit(
                historical_window[[sentiment_signal_col]],
                historical_window[target_col]
            )

            beta = beta_model.coef_[0] * 252

            S0 = test_month["Close"].iloc[0]
            steps = len(test_month)

            sentiment_signal = test_month[sentiment_signal_col].values

            n_sims = 1000
            dt = 1 / 252
            simulations = np.zeros((steps, n_sims))

            for sim in range(n_sims):
                prices = [S0]

                for t in range(1, steps):
                    Z = np.random.normal()

                    mu_adjusted = mu + beta * sentiment_signal[t]

                    St = prices[-1] * np.exp(
                        (mu_adjusted - 0.5 * sigma**2) * dt
                        + sigma * np.sqrt(dt) * Z
                    )

                    prices.append(St)

                simulations[:, sim] = prices

            gbm_mean = simulations.mean(axis=1)
            lower = np.percentile(simulations, 5, axis=1)
            upper = np.percentile(simulations, 95, axis=1)

            actual = test_month["Close"].values

            mae = np.mean(np.abs(actual - gbm_mean))
            rmse = np.sqrt(np.mean((actual - gbm_mean) ** 2))

            gbm_periodic_sentiment_results.append({
                "ticker": ticker,
                "year": year,
                "month": month,
                "window_size": window_size,
                "mu": mu,
                "sigma": sigma,
                "beta_sentiment": beta,
                "MAE": mae,
                "RMSE": rmse
            })

            temp_pred = pd.DataFrame({
                "Date": test_month["date"].values,
                "ticker": ticker,
                "year": year,
                "month": month,
                "actual": actual,
                "prediction": gbm_mean,
                "lower_5": lower,
                "upper_95": upper,
                "mu": mu,
                "sigma": sigma,
                "beta_sentiment": beta,
                "sentiment_signal": sentiment_signal
            })

            gbm_periodic_sentiment_predictions.append(temp_pred)

            ax.plot(
                test_month["date"].dt.day,
                actual,
                color="black",
                linewidth=2,
                label="Actual"
            )

            ax.plot(
                test_month["date"].dt.day,
                gbm_mean,
                color="green",
                linewidth=1.5,
                label="Periodic GBM + Sentiment Mean"
            )

            ax.fill_between(
                test_month["date"].dt.day,
                lower,
                upper,
                color="green",
                alpha=0.15,
                label="5%-95% Band"
            )

            ax.set_title(month_names[month - 1], fontsize=11, pad=8)
            ax.set_xlabel("Day")
            ax.set_ylabel("Price")
            ax.grid(True, linestyle="--", alpha=0.4)

            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1.1)

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.94),
            ncol=3,
            frameon=True
        )

        fig.suptitle(
            f"{ticker}: Periodic GBM + Sentiment Month-by-Month Forecast ({year})",
            fontsize=18,
            fontweight="bold",
            y=0.985
        )

        plt.subplots_adjust(
            top=0.88,
            hspace=0.50,
            wspace=0.30
        )

        plt.show()

gbm_periodic_sentiment_results_df = pd.DataFrame(gbm_periodic_sentiment_results)
gbm_periodic_sentiment_predictions_df = pd.concat(
    gbm_periodic_sentiment_predictions,
    ignore_index=True
)

gbm_periodic_sentiment_results_df.head()

Summary table

In [ ]:
gbm_periodic_sentiment_summary = gbm_periodic_sentiment_results_df.groupby("ticker").agg(
    avg_periodic_sent_mae=("MAE", "mean"),
    avg_periodic_sent_rmse=("RMSE", "mean"),
    best_periodic_sent_mae=("MAE", "min"),
    worst_periodic_sent_mae=("MAE", "max"),
    avg_mu=("mu", "mean"),
    avg_sigma=("sigma", "mean"),
    avg_beta_sentiment=("beta_sentiment", "mean")
).reset_index()

gbm_periodic_sentiment_summary

Compare GBM + Sentiment vs Periodic GBM + Sentiment

In [ ]:
gbm_sentiment_vs_periodic_sentiment = gbm_sentiment_monthly_summary.merge(
    gbm_periodic_sentiment_summary,
    on="ticker",
    how="inner"
)

gbm_sentiment_vs_periodic_sentiment["mae_change"] = (
    gbm_sentiment_vs_periodic_sentiment["avg_gbm_sent_monthly_mae"]
    - gbm_sentiment_vs_periodic_sentiment["avg_periodic_sent_mae"]
)

gbm_sentiment_vs_periodic_sentiment["rmse_change"] = (
    gbm_sentiment_vs_periodic_sentiment["avg_gbm_sent_monthly_rmse"]
    - gbm_sentiment_vs_periodic_sentiment["avg_periodic_sent_rmse"]
)

gbm_sentiment_vs_periodic_sentiment["periodic_sentiment_improved_mae"] = (
    gbm_sentiment_vs_periodic_sentiment["mae_change"] > 0
)

gbm_sentiment_vs_periodic_sentiment["periodic_sentiment_improved_rmse"] = (
    gbm_sentiment_vs_periodic_sentiment["rmse_change"] > 0
)

gbm_sentiment_vs_periodic_sentiment[[
    "ticker",
    "avg_gbm_sent_monthly_mae",
    "avg_periodic_sent_mae",
    "mae_change",
    "periodic_sentiment_improved_mae",
    "avg_gbm_sent_monthly_rmse",
    "avg_periodic_sent_rmse",
    "rmse_change",
    "periodic_sentiment_improved_rmse",
    "avg_beta_sentiment"
]]

Save outputs

In [ ]:
import os

save_path = "/content/drive/MyDrive/FYP_outputs"
os.makedirs(save_path, exist_ok=True)

gbm_periodic_sentiment_results_df.to_csv(
    f"{save_path}/gbm_periodic_sentiment_results.csv",
    index=False
)

gbm_periodic_sentiment_predictions_df.to_csv(
    f"{save_path}/gbm_periodic_sentiment_predictions.csv",
    index=False
)

gbm_periodic_sentiment_summary.to_csv(
    f"{save_path}/gbm_periodic_sentiment_summary.csv",
    index=False
)

gbm_sentiment_vs_periodic_sentiment.to_csv(
    f"{save_path}/gbm_sentiment_vs_periodic_sentiment.csv",
    index=False
)

print("Saved Periodic GBM + Sentiment outputs.")

##Final Model Comparison

Check required tables exist

In [ ]:
required_vars = [
    # One-step-ahead models
    "lstm_sentiment_full_comparison_df",

    # GBM models
    "gbm_monthly_summary",
    "best_window_by_ticker",
    "gbm_sentiment_monthly_summary",
    "gbm_periodic_sentiment_summary"
]

for var in required_vars:
    print(var, "exists:", var in globals())

One-step-ahead full table

In [ ]:
one_step_models = [
    "Random Walk",
    "Linear Regression",
    "LSTM",
    "LSTM + Sentiment"
]

one_step_comparison = lstm_sentiment_full_comparison_df[
    lstm_sentiment_full_comparison_df["model"].isin(one_step_models)
].copy()

one_step_comparison = one_step_comparison[[
    "ticker",
    "model",
    "MAE",
    "RMSE",
    "MAPE",
    "sMAPE",
    "MASE",
    "Mean_Error",
    "Directional_Accuracy"
]]

one_step_comparison = one_step_comparison.sort_values(
    ["ticker", "MAE"]
).reset_index(drop=True)

one_step_comparison

One-step-ahead average comparison

In [ ]:
one_step_average = one_step_comparison.groupby("model").agg(
    avg_MAE=("MAE", "mean"),
    avg_RMSE=("RMSE", "mean"),
    avg_MAPE=("MAPE", "mean"),
    avg_sMAPE=("sMAPE", "mean"),
    avg_MASE=("MASE", "mean"),
    avg_Mean_Error=("Mean_Error", "mean"),
    avg_Directional_Accuracy=("Directional_Accuracy", "mean")
).reset_index()

one_step_average = one_step_average.sort_values("avg_MAE").reset_index(drop=True)

one_step_average

Best one-step model by ticker

In [ ]:
one_step_best_by_ticker = (
    one_step_comparison
    .sort_values(["ticker", "MAE"])
    .groupby("ticker")
    .first()
    .reset_index()
)

one_step_best_by_ticker

Count one-step model wins

In [ ]:
one_step_model_wins = (
    one_step_best_by_ticker["model"]
    .value_counts()
    .reset_index()
)

one_step_model_wins.columns = ["model", "number_of_ticker_wins"]

one_step_model_wins

**GBM path-based comparison**

This compares:

Basic Monthly GBM

Periodic GBM

GBM + Sentiment

Periodic GBM + Sentiment

GBM path-based full table

In [ ]:
gbm_path_rows = []

# Basic Monthly GBM
for _, row in gbm_monthly_summary.iterrows():
    gbm_path_rows.append({
        "ticker": row["ticker"],
        "model": "Basic Monthly GBM",
        "MAE": row["avg_monthly_mae"],
        "RMSE": row["avg_monthly_rmse"]
    })

# Periodic GBM
for _, row in best_window_by_ticker.iterrows():
    gbm_path_rows.append({
        "ticker": row["ticker"],
        "model": "Periodic GBM",
        "MAE": row["avg_mae"],
        "RMSE": row["avg_rmse"]
    })

# GBM + Sentiment
for _, row in gbm_sentiment_monthly_summary.iterrows():
    gbm_path_rows.append({
        "ticker": row["ticker"],
        "model": "GBM + Sentiment",
        "MAE": row["avg_gbm_sent_monthly_mae"],
        "RMSE": row["avg_gbm_sent_monthly_rmse"]
    })

# Periodic GBM + Sentiment
for _, row in gbm_periodic_sentiment_summary.iterrows():
    gbm_path_rows.append({
        "ticker": row["ticker"],
        "model": "Periodic GBM + Sentiment",
        "MAE": row["avg_periodic_sent_mae"],
        "RMSE": row["avg_periodic_sent_rmse"]
    })

gbm_path_comparison = pd.DataFrame(gbm_path_rows)

gbm_path_comparison = gbm_path_comparison.sort_values(
    ["ticker", "MAE"]
).reset_index(drop=True)

gbm_path_comparison

GBM path-based average comparison

In [ ]:
gbm_path_average = gbm_path_comparison.groupby("model").agg(
    avg_MAE=("MAE", "mean"),
    avg_RMSE=("RMSE", "mean")
).reset_index()

gbm_path_average = gbm_path_average.sort_values("avg_MAE").reset_index(drop=True)

gbm_path_average

Best GBM model by ticker

In [ ]:
gbm_path_best_by_ticker = (
    gbm_path_comparison
    .sort_values(["ticker", "MAE"])
    .groupby("ticker")
    .first()
    .reset_index()
)

gbm_path_best_by_ticker

Count GBM model wins

In [ ]:
gbm_path_model_wins = (
    gbm_path_best_by_ticker["model"]
    .value_counts()
    .reset_index()
)

gbm_path_model_wins.columns = ["model", "number_of_ticker_wins"]

gbm_path_model_wins

**Overall combined comparison**

This table combines all models together, but we will label the forecast type clearly.

Build combined comparison table

In [ ]:
# Add forecast type labels to one-step models
one_step_for_combined = one_step_comparison.copy()
one_step_for_combined["forecast_type"] = "One-step-ahead next-day forecast"

# Add missing columns for consistency
one_step_for_combined = one_step_for_combined[[
    "ticker",
    "model",
    "forecast_type",
    "MAE",
    "RMSE",
    "MAPE",
    "sMAPE",
    "MASE",
    "Mean_Error",
    "Directional_Accuracy"
]]

# Add forecast type labels to GBM models
gbm_for_combined = gbm_path_comparison.copy()
gbm_for_combined["forecast_type"] = "GBM simulated price-path forecast"

# Add columns not available for GBM comparison
gbm_for_combined["MAPE"] = np.nan
gbm_for_combined["sMAPE"] = np.nan
gbm_for_combined["MASE"] = np.nan
gbm_for_combined["Mean_Error"] = np.nan
gbm_for_combined["Directional_Accuracy"] = np.nan

gbm_for_combined = gbm_for_combined[[
    "ticker",
    "model",
    "forecast_type",
    "MAE",
    "RMSE",
    "MAPE",
    "sMAPE",
    "MASE",
    "Mean_Error",
    "Directional_Accuracy"
]]

# Combine
final_grouped_model_comparison = pd.concat(
    [one_step_for_combined, gbm_for_combined],
    ignore_index=True
)

final_grouped_model_comparison = final_grouped_model_comparison.sort_values(
    ["forecast_type", "ticker", "MAE"]
).reset_index(drop=True)

final_grouped_model_comparison

Overall average comparison

In [ ]:
final_grouped_average = final_grouped_model_comparison.groupby(
    ["forecast_type", "model"]
).agg(
    avg_MAE=("MAE", "mean"),
    avg_RMSE=("RMSE", "mean"),
    avg_MAPE=("MAPE", "mean"),
    avg_sMAPE=("sMAPE", "mean"),
    avg_MASE=("MASE", "mean"),
    avg_Mean_Error=("Mean_Error", "mean"),
    avg_Directional_Accuracy=("Directional_Accuracy", "mean")
).reset_index()

final_grouped_average = final_grouped_average.sort_values(
    ["forecast_type", "avg_MAE"]
).reset_index(drop=True)

final_grouped_average

Overall best model by ticker within each forecast type

In [ ]:
best_model_by_ticker_and_type = (
    final_grouped_model_comparison
    .sort_values(["forecast_type", "ticker", "MAE"])
    .groupby(["forecast_type", "ticker"])
    .first()
    .reset_index()
)

best_model_by_ticker_and_type

One-step-ahead MAE plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models_order = [
    "Random Walk",
    "Linear Regression",
    "LSTM",
    "LSTM + Sentiment"
]

tickers_order = tickers
x = np.arange(len(tickers_order))
width = 0.18

plt.figure(figsize=(18, 7))

for i, model in enumerate(models_order):
    temp = one_step_comparison[one_step_comparison["model"] == model].set_index("ticker")
    values = [temp.loc[ticker, "MAE"] if ticker in temp.index else np.nan for ticker in tickers_order]

    plt.bar(
        x + i * width,
        values,
        width,
        label=model
    )

plt.xticks(
    x + width * (len(models_order) - 1) / 2,
    tickers_order
)

plt.ylabel("MAE")
plt.xlabel("Ticker")
plt.title("One-Step-Ahead Model Comparison by MAE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

One-step-ahead RMSE plot

In [ ]:
plt.figure(figsize=(18, 7))

for i, model in enumerate(models_order):
    temp = one_step_comparison[one_step_comparison["model"] == model].set_index("ticker")
    values = [temp.loc[ticker, "RMSE"] if ticker in temp.index else np.nan for ticker in tickers_order]

    plt.bar(
        x + i * width,
        values,
        width,
        label=model
    )

plt.xticks(
    x + width * (len(models_order) - 1) / 2,
    tickers_order
)

plt.ylabel("RMSE")
plt.xlabel("Ticker")
plt.title("One-Step-Ahead Model Comparison by RMSE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

GBM path-based MAE plot

In [ ]:
gbm_models_order = [
    "Basic Monthly GBM",
    "Periodic GBM",
    "GBM + Sentiment",
    "Periodic GBM + Sentiment"
]

x = np.arange(len(tickers_order))
width = 0.18

plt.figure(figsize=(18, 7))

for i, model in enumerate(gbm_models_order):
    temp = gbm_path_comparison[gbm_path_comparison["model"] == model].set_index("ticker")
    values = [temp.loc[ticker, "MAE"] if ticker in temp.index else np.nan for ticker in tickers_order]

    plt.bar(
        x + i * width,
        values,
        width,
        label=model
    )

plt.xticks(
    x + width * (len(gbm_models_order) - 1) / 2,
    tickers_order
)

plt.ylabel("MAE")
plt.xlabel("Ticker")
plt.title("GBM Path-Based Model Comparison by MAE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

GBM path-based RMSE plot

In [ ]:
plt.figure(figsize=(18, 7))

for i, model in enumerate(gbm_models_order):
    temp = gbm_path_comparison[gbm_path_comparison["model"] == model].set_index("ticker")
    values = [temp.loc[ticker, "RMSE"] if ticker in temp.index else np.nan for ticker in tickers_order]

    plt.bar(
        x + i * width,
        values,
        width,
        label=model
    )

plt.xticks(
    x + width * (len(gbm_models_order) - 1) / 2,
    tickers_order
)

plt.ylabel("RMSE")
plt.xlabel("Ticker")
plt.title("GBM Path-Based Model Comparison by RMSE")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

Save comparison tables

In [ ]:
import os

save_path = "/content/drive/MyDrive/FYP_outputs"
os.makedirs(save_path, exist_ok=True)

one_step_comparison.to_csv(
    f"{save_path}/one_step_model_comparison.csv",
    index=False
)

one_step_average.to_csv(
    f"{save_path}/one_step_average_comparison.csv",
    index=False
)

one_step_best_by_ticker.to_csv(
    f"{save_path}/one_step_best_by_ticker.csv",
    index=False
)

one_step_model_wins.to_csv(
    f"{save_path}/one_step_model_wins.csv",
    index=False
)

gbm_path_comparison.to_csv(
    f"{save_path}/gbm_path_model_comparison.csv",
    index=False
)

gbm_path_average.to_csv(
    f"{save_path}/gbm_path_average_comparison.csv",
    index=False
)

gbm_path_best_by_ticker.to_csv(
    f"{save_path}/gbm_path_best_by_ticker.csv",
    index=False
)

gbm_path_model_wins.to_csv(
    f"{save_path}/gbm_path_model_wins.csv",
    index=False
)

final_grouped_model_comparison.to_csv(
    f"{save_path}/final_grouped_model_comparison.csv",
    index=False
)

final_grouped_average.to_csv(
    f"{save_path}/final_grouped_average_comparison.csv",
    index=False
)

best_model_by_ticker_and_type.to_csv(
    f"{save_path}/best_model_by_ticker_and_type.csv",
    index=False
)

print("Saved all final comparison outputs.")

## Testing

In [ ]:
required_vars = [
    "tickers",
    "splits",
    "feature_cols",
    "target_col",
    "lstm_predictions_df",
    "lstm_sentiment_predictions_df",
    "evaluate_forecast"
]

for var in required_vars:
    print(var, "exists:", var in globals())

Build aligned one-step prediction table

In [ ]:
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np

one_step_predictions = []

for ticker in tickers:

    # Use LSTM + Sentiment dates as the common aligned evaluation dates
    sent_temp = lstm_sentiment_predictions_df[
        lstm_sentiment_predictions_df["ticker"] == ticker
    ].copy()

    sent_temp["Date"] = pd.to_datetime(sent_temp["Date"])
    sent_temp = sent_temp.sort_values("Date").reset_index(drop=True)

    aligned_dates = sent_temp["Date"]

    current_price = sent_temp["current_price"].values
    actual = sent_temp["actual"].values

    # ----------------------------
    # Random Walk
    # ----------------------------
    rw_pred = current_price

    temp_rw = pd.DataFrame({
        "Date": aligned_dates,
        "ticker": ticker,
        "model": "Random Walk",
        "current_price": current_price,
        "actual": actual,
        "prediction": rw_pred
    })

    one_step_predictions.append(temp_rw)

    # ----------------------------
    # Linear Regression
    # ----------------------------
    train = splits[ticker]["train"].copy()
    test = splits[ticker]["test"].copy()

    train["Date"] = pd.to_datetime(train["Date"])
    test["Date"] = pd.to_datetime(test["Date"])

    test_aligned = (
        test
        .set_index("Date")
        .reindex(aligned_dates)
        .reset_index()
        .rename(columns={"index": "Date"})
    )

    assert test_aligned[feature_cols].isna().sum().sum() == 0, f"{ticker}: missing aligned LR features"

    lr_model = LinearRegression()
    lr_model.fit(train[feature_cols], train[target_col])

    lr_pred_return = lr_model.predict(test_aligned[feature_cols])
    lr_pred_price = current_price * np.exp(lr_pred_return)

    temp_lr = pd.DataFrame({
        "Date": aligned_dates,
        "ticker": ticker,
        "model": "Linear Regression",
        "current_price": current_price,
        "actual": actual,
        "prediction": lr_pred_price
    })

    one_step_predictions.append(temp_lr)

    # ----------------------------
    # LSTM without sentiment
    # ----------------------------
    lstm_temp = lstm_predictions_df[
        lstm_predictions_df["ticker"] == ticker
    ].copy()

    lstm_temp["Date"] = pd.to_datetime(lstm_temp["Date"])

    lstm_aligned = (
        lstm_temp
        .set_index("Date")
        .reindex(aligned_dates)
        .reset_index()
        .rename(columns={"index": "Date"})
    )

    assert lstm_aligned["prediction"].isna().sum() == 0, f"{ticker}: missing aligned LSTM predictions"

    temp_lstm = pd.DataFrame({
        "Date": aligned_dates,
        "ticker": ticker,
        "model": "LSTM",
        "current_price": current_price,
        "actual": actual,
        "prediction": lstm_aligned["prediction"].values
    })

    one_step_predictions.append(temp_lstm)

    # ----------------------------
    # LSTM + Sentiment
    # ----------------------------
    temp_lstm_sent = pd.DataFrame({
        "Date": aligned_dates,
        "ticker": ticker,
        "model": "LSTM + Sentiment",
        "current_price": current_price,
        "actual": actual,
        "prediction": sent_temp["prediction"].values
    })

    one_step_predictions.append(temp_lstm_sent)


one_step_predictions_df = pd.concat(one_step_predictions, ignore_index=True)

one_step_predictions_df["error"] = (
    one_step_predictions_df["prediction"] -
    one_step_predictions_df["actual"]
)

one_step_predictions_df["absolute_error"] = np.abs(one_step_predictions_df["error"])
one_step_predictions_df["squared_error"] = one_step_predictions_df["error"] ** 2

one_step_predictions_df.head()

Check prediction table is clean

In [ ]:
print("Shape:", one_step_predictions_df.shape)

print("\nModels:")
print(one_step_predictions_df["model"].unique())

print("\nTickers:")
print(one_step_predictions_df["ticker"].unique())

print("\nMissing values:")
print(one_step_predictions_df.isna().sum())

Diebold-Mariano style test function

In [ ]:
from scipy import stats
import numpy as np

def diebold_mariano_test(errors_model_a, errors_model_b, loss="squared"):
    errors_model_a = np.array(errors_model_a)
    errors_model_b = np.array(errors_model_b)

    if loss == "squared":
        loss_a = errors_model_a ** 2
        loss_b = errors_model_b ** 2
    elif loss == "absolute":
        loss_a = np.abs(errors_model_a)
        loss_b = np.abs(errors_model_b)
    else:
        raise ValueError("loss must be either 'squared' or 'absolute'")

    loss_diff = loss_a - loss_b

    mean_diff = np.mean(loss_diff)
    std_diff = np.std(loss_diff, ddof=1)
    n = len(loss_diff)

    if std_diff == 0:
        return np.nan, np.nan

    dm_stat = mean_diff / (std_diff / np.sqrt(n))
    p_value = 2 * (1 - stats.t.cdf(np.abs(dm_stat), df=n - 1))

    return dm_stat, p_value

Run tests against Random Walk

In [ ]:
dm_results = []

comparison_models = [
    "Linear Regression",
    "LSTM",
    "LSTM + Sentiment"
]

for ticker in tickers:

    ticker_data = one_step_predictions_df[
        one_step_predictions_df["ticker"] == ticker
    ].copy()

    rw_errors = ticker_data[
        ticker_data["model"] == "Random Walk"
    ].sort_values("Date")["error"].values

    for model in comparison_models:

        model_errors = ticker_data[
            ticker_data["model"] == model
        ].sort_values("Date")["error"].values

        assert len(model_errors) == len(rw_errors), f"{ticker} {model}: length mismatch"

        dm_sq, p_sq = diebold_mariano_test(
            model_errors,
            rw_errors,
            loss="squared"
        )

        dm_abs, p_abs = diebold_mariano_test(
            model_errors,
            rw_errors,
            loss="absolute"
        )

        dm_results.append({
            "ticker": ticker,
            "model_vs_random_walk": model,
            "dm_stat_squared_loss": dm_sq,
            "p_value_squared_loss": p_sq,
            "significant_squared_5pct": p_sq < 0.05,
            "dm_stat_absolute_loss": dm_abs,
            "p_value_absolute_loss": p_abs,
            "significant_absolute_5pct": p_abs < 0.05
        })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df

Add interpretation labels

In [ ]:
def interpret_dm(row, loss_type):
    if loss_type == "squared":
        dm_col = "dm_stat_squared_loss"
        p_col = "p_value_squared_loss"
    else:
        dm_col = "dm_stat_absolute_loss"
        p_col = "p_value_absolute_loss"

    if row[p_col] >= 0.05:
        return "No significant difference vs Random Walk"

    if row[dm_col] < 0:
        return "Significantly better than Random Walk"

    if row[dm_col] > 0:
        return "Significantly worse than Random Walk"

    return "No clear result"


dm_results_df["squared_loss_interpretation"] = dm_results_df.apply(
    lambda row: interpret_dm(row, "squared"),
    axis=1
)

dm_results_df["absolute_loss_interpretation"] = dm_results_df.apply(
    lambda row: interpret_dm(row, "absolute"),
    axis=1
)

dm_results_df

Summary of significance results

In [ ]:
dm_summary = dm_results_df.groupby(
    ["model_vs_random_walk", "squared_loss_interpretation"]
).size().reset_index(name="count")

dm_summary

Directional accuracy significance test

In [ ]:
from scipy.stats import binomtest

directional_test_results = []

direction_models = [
    "Linear Regression",
    "LSTM",
    "LSTM + Sentiment"
]

for ticker in tickers:

    ticker_data = one_step_predictions_df[
        one_step_predictions_df["ticker"] == ticker
    ].copy()

    for model in direction_models:

        temp = ticker_data[
            ticker_data["model"] == model
        ].sort_values("Date").copy()

        actual_direction = np.sign(temp["actual"].values - temp["current_price"].values)
        predicted_direction = np.sign(temp["prediction"].values - temp["current_price"].values)

        correct = np.sum(actual_direction == predicted_direction)
        total = len(temp)

        test_result = binomtest(
            k=correct,
            n=total,
            p=0.5,
            alternative="two-sided"
        )

        directional_test_results.append({
            "ticker": ticker,
            "model": model,
            "correct_direction_count": correct,
            "total_predictions": total,
            "directional_accuracy": correct / total,
            "p_value_vs_50pct": test_result.pvalue,
            "significant_vs_50pct": test_result.pvalue < 0.05
        })

directional_test_results_df = pd.DataFrame(directional_test_results)
directional_test_results_df

Directional test summary

In [ ]:
directional_test_summary = directional_test_results_df.groupby(
    ["model", "significant_vs_50pct"]
).size().reset_index(name="count")

directional_test_summary

Save significance outputs

In [ ]:
import os

save_path = "/content/drive/MyDrive/FYP_outputs"
os.makedirs(save_path, exist_ok=True)

one_step_predictions_df.to_csv(
    f"{save_path}/one_step_predictions_aligned.csv",
    index=False
)

dm_results_df.to_csv(
    f"{save_path}/diebold_mariano_results_vs_random_walk.csv",
    index=False
)

directional_test_results_df.to_csv(
    f"{save_path}/directional_accuracy_significance_results.csv",
    index=False
)

print("Saved statistical significance outputs.")